# 5주차 — **데이터 시각화**(Data Visualization)와 **탐색적 데이터 분석**(EDA)

> **통합 완성본** — 이론("왜?"+AI연결) + 가이드 실습 + 프로젝트 + XAI·PCA 미리보기 + 연습문제 12개

---

## 학습 목표

| # | 목표 | 수준 |
|---|---|---|
| 1 | **앤스콤의 사중주**(Anscombe's Quartet)를 통해 "숫자만으로는 부족하다"는 것을 체감한다 | 이론 |
| 2 | **분포·이상치·상관관계**가 ML 성능에 미치는 영향을 "왜?"의 관점에서 이해한다 | 이론+AI |
| 3 | Seaborn·Plotly로 **7가지 핵심 그래프**를 그리고 해석한다 | 실습 |
| 4 | Airbnb NYC 데이터로 **6단계 EDA 파이프라인**을 수행한다 | 프로젝트 |
| 5 | **XAI·PCA·t-SNE**의 개념을 이해하고, EDA와 학습 후 시각화의 관계를 파악한다 | 심화 |

> → **AI 연결**: 이 수업의 모든 개념은 GPT, CNN, 추천 시스템 등 최신 AI의 **전처리·해석·디버깅**에 직결된다.

---

### 비유로 먼저 이해하기

| 비유 | 의미 |
|---|---|
| **건강검진** = EDA | 의사가 환자를 진찰하듯, 데이터를 "진찰"하는 과정이다 |
| **체온계** = 통계량 | 숫자 하나(36.5°C)만으로는 환자의 상태를 모른다 |
| 🩻 **X-ray** = 시각화 | 눈으로 직접 봐야 진짜 문제를 발견한다 |
| **돋보기** = 인터랙티브 | 확대·축소하며 세부를 탐색한다 |


---

## 환경 설정


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 라이브러리 설치 및 임포트
# ═══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── 한글 폰트 설정 ──
try:
 plt.rcParams['font.family'] = 'NanumGothic'
except:
 plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# ── 색맹 안전 팔레트 전역 설정 ──
CVD_SAFE = ['#0072B2', '#E69F00', '#009E73', '#CC79A7', '#56B4E9', '#D55E00', '#F0E442', '#000000']
sns.set_palette(CVD_SAFE)
sns.set_style('whitegrid')

print("=" * 60)
print(" ✅ 환경 설정 완료")
print("=" * 60)
print(f" NumPy : {np.__version__}")
print(f" Pandas : {pd.__version__}")
print(f" Seaborn : {sns.__version__}")
print(f" 색맹 안전 팔레트: {len(CVD_SAFE)}색 적용")
print()
print(" ▶ 비유: 화가가 그림을 그리기 전에 물감과 붓을 준비하듯,")
print(" 데이터 시각화도 '도구 준비'가 첫 번째 단계이다.")
print()
print(" → AI 연결: 실제 AI 프로젝트에서도 가장 먼저")
print(" 라이브러리 버전 호환성을 확인한다. (의존성 관리)")
print("=" * 60)


---

# PART 1: 이론 — "왜 시각화인가?"

| 기존 (도구 수준) | 보강 (전문가 수준 + AI 연결) |
|---|---|
| "`sns.histplot()`으로 히스토그램을 그린다" | **왜** 분포를 봐야 하는가? → ML 알고리즘의 전제 조건 |
| "이상치를 제거한다" | **항상** 제거해야 하는가? → 사기 탐지에서 이상치가 "답"이다 |
| "히트맵으로 상관관계를 본다" | **상관 ≠ 인과.** 다중공선성 함정은? |

> **핵심 비유**: 통계량만 보는 것은 **전화 통화**로 진료하는 것과 같다. 시각화는 **직접 대면 진료**이다.


---

## 1.1 왜 시각화인가? — **앤스콤의 사중주**(Anscombe's Quartet)

1973년, 통계학자 **프랭크 앤스콤**(Frank Anscombe)은 충격적인 데이터를 발표하였다.
네 개의 데이터셋이 있는데, **통계량이 완전히 동일**하다.

| 통계량 | 데이터셋 I | II | III | IV |
|---|---|---|---|---|
| x 평균 | 9.0 | 9.0 | 9.0 | 9.0 |
| y 평균 | 7.50 | 7.50 | 7.50 | 7.50 |
| 상관계수 r | 0.816 | 0.816 | 0.816 | 0.816 |
| 회귀선 | y = 0.5x + 3 | 동일 | 동일 | 동일 |

### ▶ 비유: "**쌍둥이 건강검진**"

> 네 명의 환자가 있다. 체온(36.5°C), 혈압(120/80), 심박수(72bpm)가 **모두 동일**하다.
> 그런데 X-ray를 찍어 보면? 한 명은 건강하고, 한 명은 폐렴이고, 한 명은 갈비뼈가 부러져 있다.
> **숫자만 보면 같아 보이지만, 그림을 보면 완전히 다르다.** 이것이 앤스콤의 교훈이다.

### → AI 연결

- **CNN**(Convolutional Neural Network)은 이미지 데이터를 "보는" 능력이 있다
- 사람도 데이터를 **시각적으로 확인**해야 모델에 잘못된 데이터를 넣는 것을 방지할 수 있다
- **데이터 중심 AI**(Data-Centric AI): Andrew Ng은 "코드보다 데이터 품질이 중요하다"고 강조한다


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-A1] ▶ 비유 시각화: "건강검진 비유" — 숫자 같아도 상태가 다르다
# ═══════════════════════════════════════════════════════════════
# 비유: 건강검진 결과지를 생각해 보자.
# 환자 A: 혈압 120, 맥박 72, 체온 36.5 → "정상"
# 환자 B: 혈압 120, 맥박 72, 체온 36.5 → "정상"
# 하지만 A는 건강하고, B는 심장에 문제가 있을 수 있다.
# "수치"만 보면 둘 다 정상이지만, "심전도 그래프"를 보면 차이가 드러난다.
# 이것이 바로 앤스콤이 말한 것이다: "숫자만 보지 말고, 그래프를 그려라!"

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("=" * 70)
print(" ▶ 비유: 건강검진 — 수치는 같지만 상태는 다르다")
print("=" * 70)
print()
print(" 환자 A와 환자 B의 건강검진 수치:")
print(" ┌──────────┬──────────┬──────────┬──────────┐")
print(" │ 항목 │ 혈압 │ 맥박 │ 체온 │")
print(" ├──────────┼──────────┼──────────┼──────────┤")
print(" │ 환자 A │ 120 │ 72 │ 36.5 │")
print(" │ 환자 B │ 120 │ 72 │ 36.5 │")
print(" └──────────┴──────────┴──────────┴──────────┘")
print(" → 수치만 보면 둘 다 '정상'이다.")
print()
print(" 하지만 심전도(ECG) 그래프를 보면?")
print()

# 심전도 비유 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor='white')

# 환자 A: 정상 심전도
t = np.linspace(0, 4, 1000)
normal_ecg = np.zeros_like(t)
for beat in range(5):
 center = 0.4 + beat * 0.8
 # P wave
 normal_ecg += 0.15 * np.exp(-((t - (center - 0.12))**2) / (2 * 0.01**2))
 # QRS complex
 normal_ecg += -0.1 * np.exp(-((t - (center - 0.02))**2) / (2 * 0.005**2))
 normal_ecg += 1.0 * np.exp(-((t - center)**2) / (2 * 0.008**2))
 normal_ecg += -0.2 * np.exp(-((t - (center + 0.03))**2) / (2 * 0.005**2))
 # T wave
 normal_ecg += 0.3 * np.exp(-((t - (center + 0.15))**2) / (2 * 0.02**2))

axes[0].plot(t, normal_ecg, 'k-', linewidth=1.5)
axes[0].set_title('환자 A: 정상 심전도', fontsize=14, fontweight='bold')
axes[0].set_ylabel('전압 (mV)')
axes[0].set_xlabel('시간 (초)')
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.3)
axes[0].set_ylim(-0.5, 1.5)
axes[0].text(0.5, -0.35, '✅ 규칙적인 리듬, 정상 파형', fontsize=11, 
 color='green', ha='center', transform=axes[0].transAxes)

# 환자 B: 비정상 심전도
np.random.seed(42)
abnormal_ecg = np.zeros_like(t)
intervals = [0.4, 1.0, 1.8, 2.3, 3.2] # 불규칙 간격
for center in intervals:
 abnormal_ecg += 0.1 * np.exp(-((t - (center - 0.12))**2) / (2 * 0.015**2))
 abnormal_ecg += -0.05 * np.exp(-((t - (center - 0.02))**2) / (2 * 0.005**2))
 amplitude = np.random.uniform(0.5, 1.2)
 abnormal_ecg += amplitude * np.exp(-((t - center)**2) / (2 * 0.01**2))
 abnormal_ecg += -0.15 * np.exp(-((t - (center + 0.03))**2) / (2 * 0.005**2))
 abnormal_ecg += 0.2 * np.exp(-((t - (center + 0.18))**2) / (2 * 0.025**2))
abnormal_ecg += np.random.normal(0, 0.03, len(t))

axes[1].plot(t, abnormal_ecg, 'k-', linewidth=1.5)
axes[1].set_title('환자 B: 부정맥 심전도', fontsize=14, fontweight='bold')
axes[1].set_ylabel('전압 (mV)')
axes[1].set_xlabel('시간 (초)')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.3)
axes[1].set_ylim(-0.5, 1.5)
axes[1].text(0.5, -0.35, '※ 불규칙 리듬, 비정상 파형', fontsize=11,
 color='red', ha='center', transform=axes[1].transAxes)

plt.suptitle(' 숫자(통계량)는 같지만, 그래프(시각화)를 보면 진실이 드러난다', 
 fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print(" ▶ 교훈:")
print(" ┌──────────────────────────────────────────────────┐")
print(" │ 앤스콤의 사중주 = 데이터의 '건강검진' │")
print(" │ 평균·표준편차·상관 = 혈압·맥박·체온 (숫자) │")
print(" │ 산점도 = 심전도 그래프 (시각화) │")
print(" │ │")
print(" │ → 숫자만 믿으면 '오진'한다! │")
print(" │ → 그래프를 그려야 '진짜 상태'가 보인다! │")
print(" └──────────────────────────────────────────────────┘")
print()
print(" → AI 연결:")
print(" 실제 의료 AI에서도 ECG 데이터를 시각화(CNN 입력)하여")
print(" 부정맥을 탐지한다. Apple Watch의 심전도 기능이 바로 이 원리이다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V01] 앤스콤의 사중주 — 4개 산점도
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V01] 앤스콤의 사중주 시각화")
print("=" * 60)
print()
print(" ▶ 비유: 같은 체온·혈압·심박수를 가진 4명의 환자를")
print(" X-ray로 찍어보는 것이다.")
print()

anscombe = sns.load_dataset('anscombe')

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('앤스콤의 사중주 (Anscombe\'s Quartet)\n— 통계량은 동일하나 분포는 완전히 다르다 —',
 fontsize=16, fontweight='bold', y=1.02)

titles = ['I. 선형 (Linear)', 'II. 비선형 (Nonlinear)',
 'III. 이상치 1개 (Outlier)', 'IV. 레버리지 (Leverage)']
descriptions = [
 '정상적인 선형 관계\n→ 회귀분석이 적합하다',
 '포물선 형태\n→ 단순 선형 회귀는 부적합하다',
 '이상치 하나가 통계량을 왜곡한다\n→ 이상치 탐지가 필수이다',
 '한 점이 회귀선을 지배한다\n→ 레버리지 포인트 확인이 필수이다'
]

for i, (ax, ds_name) in enumerate(zip(axes.flat, ['I', 'II', 'III', 'IV'])):
 subset = anscombe[anscombe['dataset'] == ds_name]
 ax.scatter(subset['x'], subset['y'], s=80, color=CVD_SAFE[i], edgecolors='black', linewidth=0.5, zorder=5)

 # 회귀선 추가
 slope, intercept, r, p, se = stats.linregress(subset['x'], subset['y'])
 x_line = np.linspace(subset['x'].min() - 1, subset['x'].max() + 1, 100)
 ax.plot(x_line, slope * x_line + intercept, '--', color='gray', alpha=0.7)

 ax.set_title(titles[i], fontsize=13, fontweight='bold')
 ax.set_xlabel('x')
 ax.set_ylabel('y')
 ax.set_xlim(3, 20)
 ax.set_ylim(2, 14)

 # 설명 텍스트
 ax.text(0.05, 0.95, descriptions[i], transform=ax.transAxes,
 fontsize=9, va='top', ha='left',
 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

 # 통계량 표시
 ax.text(0.95, 0.05, f'r = {r:.3f}\ny = {slope:.1f}x + {intercept:.1f}',
 transform=ax.transAxes, fontsize=9, va='bottom', ha='right',
 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

# ── 통계량 비교표 출력 ──
print()
print(" 통계량 비교 (모두 동일함을 확인하라):")
print(" " + "-" * 50)
for ds_name in ['I', 'II', 'III', 'IV']:
 subset = anscombe[anscombe['dataset'] == ds_name]
 r = stats.pearsonr(subset['x'], subset['y'])[0]
 print(f" 데이터셋 {ds_name}: x̄={subset['x'].mean():.1f}, ȳ={subset['y'].mean():.2f}, r={r:.3f}")
print(" " + "-" * 50)
print(" → 숫자만 보면 4개가 '동일한 데이터'처럼 보인다!")
print(" → 그러나 그래프를 보면 완전히 다른 데이터임을 알 수 있다.")
print()
print(" → AI 연결: CNN이 이미지를 '보는' 것처럼,")
print(" 데이터 과학자도 데이터를 '시각적으로' 확인해야 한다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V02] 앤스콤 통계량 비교 — "같은 숫자, 다른 모양"
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V02] 통계량 비교 막대 그래프")
print(" → 4개 데이터셋의 통계량이 얼마나 비슷한지 눈으로 확인한다")
print("=" * 60)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
stats_names = ['x 평균', 'y 평균', '상관계수 r', '기울기']

for ds_name in ['I', 'II', 'III', 'IV']:
 subset = anscombe[anscombe['dataset'] == ds_name]
 r_val = stats.pearsonr(subset['x'], subset['y'])[0]
 slope = stats.linregress(subset['x'], subset['y'])[0]
 values = [subset['x'].mean(), subset['y'].mean(), r_val, slope]

 for j, (ax, val, name) in enumerate(zip(axes, values, stats_names)):
 color = CVD_SAFE[['I','II','III','IV'].index(ds_name)]
 ax.bar(ds_name, val, color=color, edgecolor='black', linewidth=0.5)
 ax.set_title(name, fontweight='bold')
 ax.set_ylabel(name)

fig.suptitle('"숫자만 보면 구별할 수 없다" — 모든 막대 높이가 거의 동일하다',
 fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print()
print(" ▶ 교훈: 막대 그래프에서 4개가 거의 동일하게 보인다.")
print(" → '평균의 함정'에 빠지지 않으려면 반드시 분포를 시각화해야 한다.")
print(" → AI: 모델 평가에서도 '정확도(accuracy)'만 보면 위험하다.")
print(" → 정확도 95%라도 소수 클래스를 모두 틀릴 수 있다. (혼동행렬 시각화 필요)")


---

## 1.2 **분포**(Distribution)를 보는 이유 — ML 알고리즘의 전제 조건

"분포를 확인하라"는 것은 단순한 습관이 아니다. **많은 ML 알고리즘이 정규분포를 가정**한다.

| 알고리즘 | 정규분포 가정 | 위반 시 문제 |
|---|---|---|
| **선형 회귀**(Linear Regression) | 잔차(residual)가 정규분포 | 계수 추정치 편향, p-value 무의미 |
| **로지스틱 회귀**(Logistic Regression) | 특성이 극단적 왜도 없음 | 수렴 실패, 특성 중요도 왜곡 |
| **k-NN** | 거리 기반 → 스케일 민감 | 한 특성이 거리를 지배 |
| **SVM** | 커널 함수가 정규 가정 | 결정 경계 왜곡 |

### ▶ 비유: "**시험 점수 분포**"

> 반 학생 30명의 시험 점수를 생각하자.
>
> - **정규분포**: 대부분 70~80점, 양쪽으로 갈수록 줄어든다 → "보통 시험"
> - **오른쪽 꼬리**(Right-skewed): 대부분 낮은 점수, 소수만 높은 점수 → "어려운 시험"
> - **왼쪽 꼬리**(Left-skewed): 대부분 높은 점수 → "쉬운 시험"
> - **이봉분포**(Bimodal): 두 개의 봉우리 → "공부한 그룹 vs 안 한 그룹"
>
> ML 알고리즘은 "보통 시험" 상황을 기대한다. "어려운 시험" 데이터를 그대로 넣으면 성능이 떨어진다.

### → AI 연결

- **배치 정규화**(Batch Normalization): CNN에서 각 층의 출력 분포를 정규화한다 → 학습 속도 3~5배 향상
- **GPT의 토큰 분포**: 자연어에서 단어 빈도는 **지프의 법칙**(Zipf's Law)을 따른다 — 극도로 치우진 분포
- **이상 탐지**(Anomaly Detection): 정상 데이터의 분포를 학습하고, 그 분포에서 벗어나는 것을 탐지한다


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V03] 4가지 분포 유형 비교 — "시험 점수 비유"
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V03] 분포 유형 4가지")
print(" ▶ 비유: 시험 난이도에 따라 점수 분포가 달라지듯,")
print(" 데이터의 분포에 따라 적합한 ML 알고리즘이 달라진다.")
print("=" * 60)

np.random.seed(42)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# 정규분포
data_normal = np.random.normal(75, 10, 1000)
axes[0].hist(data_normal, bins=30, color=CVD_SAFE[0], edgecolor='white', alpha=0.8)
axes[0].set_title('정규분포 (Normal)\n"보통 시험"', fontweight='bold')
axes[0].axvline(np.mean(data_normal), color='red', linestyle='--', label=f'평균={np.mean(data_normal):.1f}')
axes[0].legend()

# 오른쪽 꼬리
data_right = np.random.exponential(20, 1000) + 30
axes[1].hist(data_right, bins=30, color=CVD_SAFE[1], edgecolor='white', alpha=0.8)
axes[1].set_title('오른쪽 꼬리 (Right-skewed)\n"어려운 시험"', fontweight='bold')
axes[1].axvline(np.mean(data_right), color='red', linestyle='--', label=f'평균={np.mean(data_right):.1f}')
axes[1].axvline(np.median(data_right), color='green', linestyle='--', label=f'중앙값={np.median(data_right):.1f}')
axes[1].legend(fontsize=8)

# 왼쪽 꼬리
data_left = 100 - np.random.exponential(15, 1000)
axes[2].hist(data_left, bins=30, color=CVD_SAFE[2], edgecolor='white', alpha=0.8)
axes[2].set_title('왼쪽 꼬리 (Left-skewed)\n"쉬운 시험"', fontweight='bold')
axes[2].axvline(np.mean(data_left), color='red', linestyle='--', label=f'평균={np.mean(data_left):.1f}')
axes[2].legend()

# 이봉분포
data_bimodal = np.concatenate([np.random.normal(40, 8, 500), np.random.normal(80, 8, 500)])
axes[3].hist(data_bimodal, bins=30, color=CVD_SAFE[3], edgecolor='white', alpha=0.8)
axes[3].set_title('이봉분포 (Bimodal)\n"공부한 vs 안 한 그룹"', fontweight='bold')

for ax in axes:
 ax.set_xlabel('점수')
 ax.set_ylabel('빈도')

plt.tight_layout()
plt.show()

print()
print(" ▶ 핵심 관찰:")
print(" 1) 정규분포: 평균 = 중앙값 → ML 알고리즘이 기대하는 형태")
print(" 2) 오른쪽 꼬리: 평균 > 중앙값 → 평균이 왜곡됨 → 로그 변환 필요")
print(" 3) 왼쪽 꼬리: 평균 < 중앙값 → 제곱/지수 변환 고려")
print(" 4) 이봉분포: 두 그룹 혼합 → 군집화(Clustering) 먼저 수행")
print()
print(" → AI: CNN의 배치 정규화(Batch Norm)는 각 층의 출력을")
print(" 정규분포로 만들어 학습을 안정화한다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V04] 왜도(Skewness)가 ML에 미치는 영향 — 변환 전후 비교
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V04] 왜도와 로그 변환")
print(" ▶ 비유: 소득 분포를 생각하라.")
print(" 대부분 연봉 3000~5000만 원이지만, 소수의 억대 연봉자가 있다.")
print(" 이 소수가 '평균'을 끌어올려 왜곡한다.")
print(" → 로그 변환으로 '압축'하면 정규분포에 가까워진다.")
print("=" * 60)

np.random.seed(42)
income = np.random.lognormal(mean=3.8, sigma=0.8, size=2000) # 소득 비유

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 원본
skew_orig = stats.skew(income)
axes[0].hist(income, bins=50, color=CVD_SAFE[0], edgecolor='white', alpha=0.8)
axes[0].set_title(f'① 원본 (왜도 = {skew_orig:.2f})\n"소수의 고소득자가 평균을 왜곡한다"', fontweight='bold')
axes[0].axvline(np.mean(income), color='red', linestyle='--', lw=2, label=f'평균 = {np.mean(income):.1f}')
axes[0].axvline(np.median(income), color='green', linestyle='--', lw=2, label=f'중앙값 = {np.median(income):.1f}')
axes[0].legend()

# 로그 변환
log_income = np.log1p(income)
skew_log = stats.skew(log_income)
axes[1].hist(log_income, bins=50, color=CVD_SAFE[2], edgecolor='white', alpha=0.8)
axes[1].set_title(f'② 로그 변환 (왜도 = {skew_log:.2f})\n"큰 값을 압축하여 균형을 맞춘다"', fontweight='bold')

# 표준화
from sklearn.preprocessing import StandardScaler
std_income = StandardScaler().fit_transform(log_income.reshape(-1, 1)).flatten()
skew_std = stats.skew(std_income)
axes[2].hist(std_income, bins=50, color=CVD_SAFE[4], edgecolor='white', alpha=0.8)
axes[2].set_title(f'③ 표준화 (왜도 = {skew_std:.2f})\n"평균=0, 표준편차=1로 정규화"', fontweight='bold')

for ax in axes:
 ax.set_ylabel('빈도')

# 화살표 추가
fig.text(0.35, 0.02, '→ log(x+1)', fontsize=14, fontweight='bold', color='red', ha='center')
fig.text(0.68, 0.02, '→ (x-μ)/σ', fontsize=14, fontweight='bold', color='red', ha='center')

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

print()
print(f" 왜도 변화: {skew_orig:.2f} → {skew_log:.2f} → {skew_std:.2f}")
print(f" → 원본 왜도 {skew_orig:.2f}는 ML 알고리즘에 문제를 일으킨다.")
print(f" → 로그 변환 후 {skew_log:.2f}로 크게 개선되었다.")
print(f" → 최종 표준화 후 {skew_std:.2f}로 정규분포에 가까워졌다.")
print()
print(" → AI 연결: GPT의 토큰 임베딩도 학습 전에 정규화(Layer Norm)를 적용한다.")
print(" 트랜스포머의 모든 층은 LayerNorm → Attention → LayerNorm → FFN 구조이다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V05] QQ-plot — 정규성 검정의 시각적 도구
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V05] QQ-plot (Quantile-Quantile Plot)")
print(" ▶ 비유: '이상적인 학생 줄 세우기'와 '실제 줄 세우기'를")
print(" 비교하여 정규분포에 얼마나 가까운지 확인하는 방법이다.")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 정규분포 데이터
normal_data = np.random.normal(0, 1, 500)
stats.probplot(normal_data, dist="norm", plot=axes[0])
axes[0].set_title('정규분포 → 직선에 가깝다', fontweight='bold')
axes[0].get_lines()[0].set_color(CVD_SAFE[0])

# 왜도가 큰 데이터 (원본)
stats.probplot(income, dist="norm", plot=axes[1])
axes[1].set_title('왜도가 큰 원본 → 곡선', fontweight='bold')
axes[1].get_lines()[0].set_color(CVD_SAFE[1])

# 로그 변환 후
stats.probplot(log_income, dist="norm", plot=axes[2])
axes[2].set_title('로그 변환 후 → 직선에 가까워짐', fontweight='bold')
axes[2].get_lines()[0].set_color(CVD_SAFE[2])

plt.tight_layout()
plt.show()

print()
print(" ▶ QQ-plot 읽는 법:")
print(" • 점들이 빨간 직선 위에 놓이면 → 정규분포이다")
print(" • 직선에서 벗어나면 → 정규분포가 아니다")
print(" • 끝부분이 위로 휘면 → 오른쪽 꼬리가 길다 (왜도 > 0)")
print()
print(" → AI: 딥러닝에서 가중치(weight) 초기화 시 정규분포를 사용한다.")
print(" He 초기화: N(0, √(2/n)) — 이 분포가 맞는지 QQ-plot으로 확인할 수 있다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V06] → AI 시각화 — 배치 정규화(Batch Normalization) 개념
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V06] CNN의 배치 정규화가 하는 일")
print(" ▶ 비유: 요리할 때 재료의 양을 표준화하는 것처럼,")
print(" 신경망도 각 층의 출력을 '표준화'한다.")
print("=" * 60)

np.random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('배치 정규화(Batch Normalization)의 효과\n— CNN의 각 층 출력 분포를 정규화한다 —',
 fontsize=14, fontweight='bold')

# 상단: BatchNorm 없이 (분포가 점점 왜곡됨)
for i, ax in enumerate(axes[0]):
 if i == 0:
 data = np.random.normal(0, 1, 1000)
 elif i == 1:
 data = np.random.normal(3, 2.5, 1000) # 평균 이동, 분산 증가
 else:
 data = np.random.exponential(2, 1000) + 5 # 심하게 왜곡
 ax.hist(data, bins=40, color=CVD_SAFE[5], edgecolor='white', alpha=0.8)
 ax.set_title(f'Layer {i+1} (BatchNorm ❌)', fontweight='bold')
 ax.set_xlim(-5, 15)
 ax.axvline(np.mean(data), color='red', linestyle='--')

# 하단: BatchNorm 있을 때 (모두 표준 정규분포)
for i, ax in enumerate(axes[1]):
 data = np.random.normal(0, 1, 1000) # 모두 정규화됨
 ax.hist(data, bins=40, color=CVD_SAFE[0], edgecolor='white', alpha=0.8)
 ax.set_title(f'Layer {i+1} (BatchNorm ✅)', fontweight='bold')
 ax.set_xlim(-5, 15)
 ax.axvline(0, color='red', linestyle='--')

plt.tight_layout()
plt.show()

print()
print(" ▶ 핵심 관찰:")
print(" • BatchNorm 없이: 층이 깊어질수록 분포가 왜곡되고 이동한다")
print(" → 이것을 '내부 공변량 이동(Internal Covariate Shift)'이라 한다")
print(" • BatchNorm 있을 때: 모든 층의 출력이 평균=0, 분산=1로 유지된다")
print(" → 학습이 안정적이고 3~5배 빠르다")
print()
print(" → AI: ResNet, EfficientNet, Vision Transformer 모두 정규화를 사용한다.")
print(" ResNet → BatchNorm, Transformer → LayerNorm")


---

## 1.3 **이상치**(Outlier) — "항상 제거해야 하는가?"

이상치에는 **두 종류**가 있다:

| 유형 | 설명 | 처리 |
|---|---|---|
| **오류**(Error) | 입력 실수, 센서 고장 | **제거** 또는 수정 |
| **희귀 사건**(Rare Event) | 사기 거래, 희귀 질병 | **절대 제거 금지** — 이것이 "답"이다! |

### ▶ 비유: "**반에서 190cm인 학생**"

> 중학교 반에서 평균 키가 160cm인데 한 학생이 190cm이다.
>
> - 만약 **키 데이터 입력 오류**(1900mm → 190cm 변환 실수)라면 → **수정해야 한다**
> - 만약 **실제로 190cm인 학생**이라면 → **제거하면 안 된다!**
> - 오히려 농구부 스카우터라면 **이 학생이 가장 중요한 데이터**이다
>
> AI에서도 마찬가지이다:
> - 신용카드 **사기 탐지**: 이상치 = 사기 거래 → 이것이 찾아야 할 "답"이다
> - 의료 AI: 이상치 = 희귀 질병 → 가장 중요한 데이터이다

### → AI 연결

- **오토인코더**(Autoencoder): 정상 데이터만 학습 → 이상치의 복원 오차가 큼 → 이상 탐지
- **Isolation Forest**: 이상치를 "고립시키는" 알고리즘 — 이상치는 빨리 분리된다
- **GAN 기반 이상 탐지**: 정상 데이터의 분포를 학습하고, 그 분포에서 벗어나면 이상으로 판단한다


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-W1] 인터랙티브: 분포 탐색기 — "분포가 바뀌면 ML이 어떻게 반응하는가?"
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "요리사의 재료 상태"
# - 정규분포 = 신선한 재료 → 어떤 요리법(알고리즘)이든 잘 된다
# - 편향된 분포 = 상한 재료 → 특정 요리법은 실패한다
# - 슬라이더를 움직여서 재료 상태(분포)를 바꿔 보라!

import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print(" 인터랙티브: 분포 탐색기")
print("=" * 70)
print()
print(" 슬라이더를 움직여 다양한 분포를 실험해 보라.")
print(" 각 분포가 ML 알고리즘에 미치는 영향을 확인할 수 있다.")
print()

out_dist = widgets.Output()

dist_type = widgets.Dropdown(
 options=['정규분포 (Normal)', '오른쪽 편향 (Right Skew)', 
 '왼쪽 편향 (Left Skew)', '이봉분포 (Bimodal)', 
 '균일분포 (Uniform)'],
 value='정규분포 (Normal)', description='분포 유형:',
 style={'description_width': '80px'}, layout=widgets.Layout(width='350px')
)
sample_slider = widgets.IntSlider(value=1000, min=100, max=5000, step=100,
 description='표본 크기:', style={'description_width': '80px'},
 layout=widgets.Layout(width='350px'))

def update_dist(*args):
 with out_dist:
 clear_output(wait=True)
 np.random.seed(42)
 n = sample_slider.value
 
 if '정규' in dist_type.value:
 data = np.random.normal(50, 15, n)
 ml_msg = "✅ 대부분의 ML 알고리즘이 잘 작동한다."
 color = '#2196F3'
 elif '오른쪽' in dist_type.value:
 data = np.random.exponential(20, n) + 10
 ml_msg = "※ 선형 회귀 성능 저하 → log 변환 필요!"
 color = '#FF5722'
 elif '왼쪽' in dist_type.value:
 data = 100 - np.random.exponential(20, n)
 ml_msg = "※ 왜도 음수 → 제곱/세제곱 변환 고려!"
 color = '#FF9800'
 elif '이봉' in dist_type.value:
 data = np.concatenate([np.random.normal(30, 8, n//2),
 np.random.normal(70, 8, n//2)])
 ml_msg = " 두 그룹 존재! → 군집 분석(K-Means) 먼저 수행!"
 color = '#9C27B0'
 else:
 data = np.random.uniform(10, 90, n)
 ml_msg = " 정보량 낮음 → 특성(feature)으로서 가치 검토 필요!"
 color = '#607D8B'
 
 fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), facecolor='white')
 
 # 히스토그램
 axes[0].hist(data, bins=40, color=color, alpha=0.7, edgecolor='white')
 axes[0].axvline(np.mean(data), color='red', linestyle='--', linewidth=2, label=f'평균: {np.mean(data):.1f}')
 axes[0].axvline(np.median(data), color='blue', linestyle='--', linewidth=2, label=f'중앙값: {np.median(data):.1f}')
 axes[0].legend(fontsize=9)
 axes[0].set_title('히스토그램', fontweight='bold')
 
 # QQ plot
 stats.probplot(data, dist="norm", plot=axes[1])
 axes[1].set_title('QQ-Plot (정규성 검정)', fontweight='bold')
 axes[1].get_lines()[0].set_color(color)
 
 # 박스플롯
 bp = axes[2].boxplot(data, vert=True, patch_artist=True)
 bp['boxes'][0].set_facecolor(color)
 bp['boxes'][0].set_alpha(0.5)
 skew = stats.skew(data)
 kurt = stats.kurtosis(data)
 axes[2].set_title(f'박스플롯\n왜도={skew:.2f}, 첨도={kurt:.2f}', fontweight='bold')
 
 plt.suptitle(f'{dist_type.value} | n={n}', fontsize=14, fontweight='bold')
 plt.tight_layout()
 plt.show()
 
 print(f"\n ML 영향: {ml_msg}")
 print(f" → AI 연결: GPT 학습 시 입력 데이터도 Layer Normalization으로")
 print(f" 분포를 정규화한다. 분포 관리는 모든 AI의 핵심이다.")

dist_type.observe(update_dist, names='value')
sample_slider.observe(update_dist, names='value')
display(widgets.VBox([
 widgets.HBox([dist_type, sample_slider]),
 out_dist
]))
update_dist()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V07] 이상치의 두 얼굴 — 오류 vs 희귀 사건
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V07] 이상치의 두 종류")
print(" ▶ 비유: 같은 '이상치'라도 오류인지 보물인지 판단해야 한다.")
print(" 농구 스카우터에게 190cm 학생은 '보물'이다!")
print("=" * 60)

np.random.seed(42)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 왼쪽: 정상 데이터 + 오류
normal_data = np.random.normal(160, 8, 100)
error_data = np.array([250, 280, 10]) # 입력 오류
all_data_err = np.concatenate([normal_data, error_data])

axes[0].hist(normal_data, bins=20, color=CVD_SAFE[0], edgecolor='white', alpha=0.7, label='정상')
axes[0].scatter(error_data, [1]*3, color='red', s=200, marker='X', zorder=5, label='오류 (제거)')
axes[0].set_title('① 오류 이상치\n입력 실수, 센서 고장', fontweight='bold')
axes[0].legend()

# 가운데: 정상 + 희귀 사건
fraud_normal = np.random.normal(50, 15, 970)
fraud_rare = np.random.normal(200, 20, 30)
axes[1].hist(fraud_normal, bins=30, color=CVD_SAFE[0], edgecolor='white', alpha=0.7, label='정상 거래')
axes[1].hist(fraud_rare, bins=10, color=CVD_SAFE[5], edgecolor='white', alpha=0.7, label='사기 거래 (보존!)')
axes[1].set_title('② 희귀 사건 이상치\n사기 탐지에서 이것이 "답"이다', fontweight='bold')
axes[1].legend()

# 오른쪽: IQR 방법 시각적 설명
box_data = np.concatenate([np.random.normal(100, 20, 200), [200, 210, 5]])
bp = axes[2].boxplot(box_data, vert=True, patch_artist=True,
 boxprops=dict(facecolor=CVD_SAFE[4], alpha=0.6),
 flierprops=dict(marker='o', markerfacecolor='red', markersize=8))
axes[2].set_title('③ IQR 방법\nQ1-1.5×IQR ~ Q3+1.5×IQR', fontweight='bold')
Q1 = np.percentile(box_data, 25)
Q3 = np.percentile(box_data, 75)
IQR = Q3 - Q1
axes[2].axhline(Q1 - 1.5*IQR, color='red', linestyle='--', alpha=0.7, label=f'하한 = {Q1-1.5*IQR:.0f}')
axes[2].axhline(Q3 + 1.5*IQR, color='red', linestyle='--', alpha=0.7, label=f'상한 = {Q3+1.5*IQR:.0f}')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print()
print(f" IQR 이상치 탐지:")
print(f" Q1 = {Q1:.1f}, Q3 = {Q3:.1f}, IQR = {IQR:.1f}")
print(f" 하한 = Q1 - 1.5×IQR = {Q1 - 1.5*IQR:.1f}")
print(f" 상한 = Q3 + 1.5×IQR = {Q3 + 1.5*IQR:.1f}")
print(f" 이상치 수 = {sum((box_data < Q1-1.5*IQR) | (box_data > Q3+1.5*IQR))}개")
print()
print(" → AI: 오토인코더(Autoencoder)는 정상 데이터만 학습한 후,")
print(" 복원 오차가 큰 데이터를 이상치로 탐지한다. (임계값 = IQR 개념과 유사)")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V08] → AI 시각화 — 오토인코더 이상 탐지 개념
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V08] 오토인코더(Autoencoder) 이상 탐지 원리")
print(" ▶ 비유: 화가가 '정상적인 풍경화'만 연습했다고 하자.")
print(" 정상 풍경을 보고 그리면 원본과 비슷하게 그리지만,")
print(" '이상한 풍경'(사기 거래)을 보고 그리면 원본과 다르게 그린다.")
print(" → 그 '차이(복원 오차)'로 이상을 탐지한다.")
print("=" * 60)

np.random.seed(42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 정상 데이터 생성
n_normal = 300
n_anomaly = 20
normal_x = np.random.normal(5, 1.5, n_normal)
normal_y = np.random.normal(5, 1.5, n_normal)

# 이상 데이터 (멀리 떨어진 점)
anomaly_x = np.random.uniform(0, 10, n_anomaly)
anomaly_y = np.random.uniform(8, 12, n_anomaly)

# 복원 오차 시뮬레이션
normal_errors = np.random.exponential(0.3, n_normal)
anomaly_errors = np.random.exponential(2.0, n_anomaly)

# 왼쪽: 데이터 분포
axes[0].scatter(normal_x, normal_y, c=CVD_SAFE[0], alpha=0.5, s=30, label='정상')
axes[0].scatter(anomaly_x, anomaly_y, c=CVD_SAFE[5], marker='X', s=100, label='이상치')
axes[0].set_title('입력 데이터 분포', fontweight='bold', fontsize=13)
axes[0].set_xlabel('특성 1')
axes[0].set_ylabel('특성 2')
axes[0].legend()

# 오른쪽: 복원 오차 분포
all_errors = np.concatenate([normal_errors, anomaly_errors])
colors = [CVD_SAFE[0]]*n_normal + [CVD_SAFE[5]]*n_anomaly
threshold = np.percentile(normal_errors, 95)
axes[1].hist(normal_errors, bins=30, color=CVD_SAFE[0], alpha=0.7, label='정상 복원 오차')
axes[1].hist(anomaly_errors, bins=15, color=CVD_SAFE[5], alpha=0.7, label='이상 복원 오차')
axes[1].axvline(threshold, color='red', linestyle='--', lw=2, label=f'임계값 = {threshold:.2f}')
axes[1].set_title('복원 오차(Reconstruction Error) 분포', fontweight='bold', fontsize=13)
axes[1].set_xlabel('복원 오차')
axes[1].set_ylabel('빈도')
axes[1].legend()

plt.tight_layout()
plt.show()

print()
print(" ▶ 오토인코더 이상 탐지 원리:")
print(" 1단계: 정상 데이터만으로 '압축 → 복원' 학습")
print(" 2단계: 새 데이터를 입력 → 복원 → 원본과 비교")
print(" 3단계: 복원 오차 > 임계값 → 이상치로 판정")
print(f" → 임계값 예시: 정상 데이터의 95% 백분위 = {threshold:.2f}")
print()
print(" → 이것은 5주차 EDA의 IQR 이상치 탐지와 동일한 논리이다!")
print(" EDA: 데이터 분포에서 벗어나는 값 탐지")
print(" AI: 학습된 분포에서 벗어나는 패턴 탐지")


---

## 1.4 **상관관계**(Correlation) — "특성 선택의 첫 단추"

상관 히트맵을 그리는 **세 가지 실전 목적**:

| 목적 | 방법 | 결과 |
|---|---|---|
| **특성 선택**(Feature Selection) | 타겟과 상관 높은 특성 선택 | 모델 성능 ↑ |
| **다중공선성**(Multicollinearity) 탐지 | 특성 간 상관 > 0.9 확인 | 불안정한 계수 방지 |
| **데이터 누수**(Data Leakage) 탐지 | 타겟과 상관 ≈ 1.0 확인 | 비현실적 성능 방지 |

### ▶ 비유: "**친구 관계도**"

> 반 학생 30명의 친밀도를 표로 그린다고 생각하자.
>
> - A와 B가 친하고, B와 C도 친하면 → A와 C도 친할 **가능성이 높다** (하지만 확실하지 않다!)
> - "키와 몸무게"는 상관이 높다 → 둘 다 넣으면 **정보 중복** (다중공선성)
> - "시험 점수와 성적"이 상관 0.99 → **데이터 누수!** (성적이 점수에서 계산되므로)
>
> **"상관 ≠ 인과"**: 아이스크림 판매량과 익사 사고는 상관이 높다.
> 하지만 아이스크림이 익사를 유발하는 것이 아니다. **숨은 변수(여름 기온)**가 있다.

### → AI 연결

- **어텐션 메커니즘**(Attention): 트랜스포머는 단어 간 "상관관계"를 학습한다
- **특성 중요도**(Feature Importance): 상관이 높은 특성부터 모델에 투입한다
- **SHAP 값**: 각 특성이 예측에 미친 "기여도"를 시각화한다


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V09] 상관관계의 4가지 유형 + "상관 ≠ 인과"
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V09] 상관관계의 4가지 유형")
print(" ▶ 비유: 친구 관계처럼 '가깝다'와 '영향을 준다'는 다르다.")
print(" A와 B가 같은 수업을 듣는다고(상관), A가 B의 성적에 영향을 주는 것은 아니다(인과).")
print("=" * 60)

np.random.seed(42)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 양의 상관 (r ≈ 0.85)
x1 = np.random.normal(0, 1, 100)
y1 = 0.85 * x1 + np.random.normal(0, 0.5, 100)
axes[0,0].scatter(x1, y1, c=CVD_SAFE[0], alpha=0.6, edgecolors='black', linewidth=0.3)
r1 = np.corrcoef(x1, y1)[0,1]
axes[0,0].set_title(f'① 강한 양의 상관 (r = {r1:.2f})\n"키 ↑ → 몸무게도 ↑"', fontweight='bold')

# 음의 상관 (r ≈ -0.80)
x2 = np.random.normal(0, 1, 100)
y2 = -0.8 * x2 + np.random.normal(0, 0.5, 100)
axes[0,1].scatter(x2, y2, c=CVD_SAFE[1], alpha=0.6, edgecolors='black', linewidth=0.3)
r2 = np.corrcoef(x2, y2)[0,1]
axes[0,1].set_title(f'② 강한 음의 상관 (r = {r2:.2f})\n"공부 시간 ↑ → 게임 시간 ↓"', fontweight='bold')

# 무상관 (r ≈ 0)
x3 = np.random.normal(0, 1, 100)
y3 = np.random.normal(0, 1, 100)
axes[1,0].scatter(x3, y3, c=CVD_SAFE[2], alpha=0.6, edgecolors='black', linewidth=0.3)
r3 = np.corrcoef(x3, y3)[0,1]
axes[1,0].set_title(f'③ 무상관 (r = {r3:.2f})\n"신발 크기 ↔ 수학 점수"', fontweight='bold')

# 비선형 상관 (r ≈ 0이지만 관계 있음!)
x4 = np.linspace(-3, 3, 100)
y4 = x4**2 + np.random.normal(0, 0.5, 100)
axes[1,1].scatter(x4, y4, c=CVD_SAFE[5], alpha=0.6, edgecolors='black', linewidth=0.3)
r4 = np.corrcoef(x4, y4)[0,1]
axes[1,1].set_title(f'④ 비선형 관계 (r = {r4:.2f} ← 함정!)\n"r≈0이지만 명확한 패턴이 있다"', fontweight='bold')
axes[1,1].annotate('※ 피어슨 r은\n비선형을 못 잡는다!',
 xy=(0, 0), fontsize=10, color='red', fontweight='bold',
 bbox=dict(boxstyle='round', facecolor='lightyellow'))

for ax in axes.flat:
 ax.set_xlabel('X')
 ax.set_ylabel('Y')
 ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print()
print(" ▶ 핵심 교훈:")
print(" • 피어슨 r은 '선형' 관계만 측정한다")
print(" • r ≈ 0이라도 비선형 관계가 있을 수 있다 (④번 참조)")
print(" • → 반드시 산점도를 그려서 확인해야 한다 (앤스콤의 교훈과 동일!)")
print()
print(" → AI 연결: 트랜스포머의 어텐션(Attention) 메커니즘은")
print(" 단어 간 '연관성 점수'를 계산한다. 이것은 상관계수의 확장판이다.")
print(" Attention(Q,K,V) = softmax(QK^T / √d) × V")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-W2] 인터랙티브: 이상치 탐지 임계값 조절기
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "보안 검색대의 민감도"
# - 임계값을 낮추면: 의심스러운 것도 다 잡는다 → 오탐(False Positive) 증가
# - 임계값을 높이면: 위험한 것만 잡는다 → 미탐(False Negative) 증가
# - 완벽한 보안(정밀도)과 편의(재현율) 사이의 트레이드오프!

import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print(" 인터랙티브: 이상치 탐지 — IQR 배수를 조절해 보라")
print("=" * 70)
print()
print(" ▶ 비유: 공항 보안 검색의 '민감도 다이얼'과 같다.")
print(" → 민감도(IQR 배수)를 낮추면 더 많은 데이터를 '이상치'로 판정한다.")
print(" → 민감도(IQR 배수)를 높이면 극단적인 것만 '이상치'로 판정한다.")
print()

out_outlier = widgets.Output()
iqr_slider = widgets.FloatSlider(value=1.5, min=0.5, max=3.0, step=0.1,
 description='IQR 배수:', style={'description_width': '80px'},
 layout=widgets.Layout(width='400px'))

np.random.seed(42)
normal_data = np.random.normal(50, 10, 200)
outliers_added = np.array([5, 8, 95, 100, 110, -10, 115])
sample_data = np.concatenate([normal_data, outliers_added])

def update_outlier(*args):
 with out_outlier:
 clear_output(wait=True)
 k = iqr_slider.value
 q1, q3 = np.percentile(sample_data, 25), np.percentile(sample_data, 75)
 iqr = q3 - q1
 lower, upper = q1 - k * iqr, q3 + k * iqr
 
 is_outlier = (sample_data < lower) | (sample_data > upper)
 n_outlier = is_outlier.sum()
 pct = n_outlier / len(sample_data) * 100
 
 fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='white')
 
 # 산점도
 ax1.scatter(range(len(sample_data)), sample_data, 
 c=['red' if o else '#2196F3' for o in is_outlier],
 alpha=0.6, s=20)
 ax1.axhline(upper, color='red', linestyle='--', linewidth=1.5, label=f'상한: {upper:.1f}')
 ax1.axhline(lower, color='red', linestyle='--', linewidth=1.5, label=f'하한: {lower:.1f}')
 ax1.fill_between(range(len(sample_data)), lower, upper, alpha=0.1, color='green')
 ax1.set_title(f'이상치 탐지 (IQR×{k:.1f})\n빨간 점 = 이상치 {n_outlier}개 ({pct:.1f}%)', 
 fontweight='bold')
 ax1.legend(fontsize=9)
 ax1.set_xlabel('데이터 인덱스')
 ax1.set_ylabel('값')
 
 # 히스토그램
 ax2.hist(sample_data[~is_outlier], bins=30, color='#2196F3', alpha=0.7, label='정상', edgecolor='white')
 ax2.hist(sample_data[is_outlier], bins=10, color='red', alpha=0.7, label='이상치', edgecolor='white')
 ax2.axvline(lower, color='red', linestyle='--', linewidth=1.5)
 ax2.axvline(upper, color='red', linestyle='--', linewidth=1.5)
 ax2.set_title('정상 vs 이상치 분포', fontweight='bold')
 ax2.legend(fontsize=10)
 
 plt.tight_layout()
 plt.show()
 
 print(f"\n 결과: Q1={q1:.1f}, Q3={q3:.1f}, IQR={iqr:.1f}")
 print(f" → 하한={lower:.1f}, 상한={upper:.1f}")
 print(f" → 이상치 {n_outlier}개 탐지 ({pct:.1f}%)")
 print()
 if k < 1.0:
 print(" ※ 너무 민감! 정상 데이터도 이상치로 오판될 수 있다.")
 elif k > 2.5:
 print(" ※ 너무 둔감! 진짜 이상치를 놓칠 수 있다.")
 else:
 print(" ✅ 적절한 범위이다. (통상 1.5가 표준)")
 print()
 print(" → AI 연결: 이상 탐지 AI(Anomaly Detection)에서도")
 print(" 임계값 설정이 핵심이다. 너무 민감하면 오탐(False Alarm),")
 print(" 너무 둔감하면 사기·해킹을 놓친다.")

iqr_slider.observe(update_outlier, names='value')
display(widgets.VBox([iqr_slider, out_outlier]))
update_outlier()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V10] → AI 시각화 — 어텐션(Attention)은 "상관관계의 확장"이다
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V10] 트랜스포머 어텐션 = 단어 간 상관관계 시각화")
print(" ▶ 비유: '나는 배가 고프다'에서 '고프다'가 '배'에 주목하는 것이")
print(" 어텐션이다. '나'보다 '배'가 더 관련 있으므로 가중치가 높다.")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 상관 히트맵 (EDA)
np.random.seed(42)
corr_data = pd.DataFrame({
 'price': np.random.normal(100, 30, 100),
 'reviews': np.random.normal(50, 15, 100),
 'rating': np.random.normal(4.0, 0.5, 100),
 'distance': np.random.normal(5, 2, 100),
})
corr_data['reviews'] = corr_data['price'] * 0.3 + np.random.normal(0, 10, 100)
corr_data['rating'] = -corr_data['distance'] * 0.5 + np.random.normal(4, 0.3, 100)

sns.heatmap(corr_data.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
 center=0, ax=axes[0], vmin=-1, vmax=1,
 linewidths=1, linecolor='white')
axes[0].set_title('EDA 상관 히트맵\n특성 간 선형 상관계수', fontweight='bold', fontsize=12)

# 오른쪽: 어텐션 히트맵 (AI)
words = ['나는', '오늘', '맛있는', '밥을', '먹었다']
attention = np.array([
 [0.8, 0.05, 0.05, 0.05, 0.05],
 [0.1, 0.6, 0.1, 0.1, 0.1],
 [0.05, 0.1, 0.3, 0.5, 0.05],
 [0.1, 0.05, 0.3, 0.4, 0.15],
 [0.15, 0.1, 0.15, 0.2, 0.4],
])

sns.heatmap(attention, annot=True, fmt='.2f', cmap='YlOrRd',
 xticklabels=words, yticklabels=words, ax=axes[1],
 linewidths=1, linecolor='white')
axes[1].set_title('트랜스포머 어텐션 히트맵\n단어 간 연관성 가중치', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Key (참조되는 단어)')
axes[1].set_ylabel('Query (주목하는 단어)')

plt.tight_layout()
plt.show()

print()
print(" ▶ 비교:")
print(" • EDA 히트맵: 특성 A와 특성 B의 '선형 상관계수' (고정된 값)")
print(" • AI 어텐션: 단어 A가 단어 B에 '주목하는 정도' (문맥에 따라 변함)")
print()
print(" ▶ '맛있는'이라는 단어는 '밥을'에 0.50의 높은 어텐션을 보인다.")
print(" → '맛있는'이 무엇을 수식하는지 어텐션이 파악한 것이다!")
print()
print(" → GPT-4는 이런 어텐션 층을 96개나 쌓아서 복잡한 문맥을 이해한다.")
print(" EDA의 상관 분석이 이해되면, 어텐션의 원리도 자연스럽게 이해된다.")


---

## 1.5 색상의 과학 — **색각 이상**(CVD) 고려

데이터 시각화에서 색상 선택은 **미적 취향이 아니라 전문적 소양**이다.

| 항목 | 내용 |
|---|---|
| **남성** 약 8%, **여성** 약 0.5%가 색각 이상(CVD)을 가진다 | 30명 중 1~2명 |
| **적녹 색맹**(Deuteranopia)이 가장 흔하다 | 빨강-초록 구분 불가 |
| **Matplotlib 기본 팔레트**는 색맹에게 불리하다 | 빨강+초록 동시 사용 |

### ▶ 비유: "**유니버설 디자인**"

> 건물에 계단만 있으면 휠체어 이용자가 들어갈 수 없다.
> **경사로**(ramp)를 추가하면 모든 사람이 이용할 수 있다.
> 마찬가지로, 색맹 안전 팔레트를 사용하면 **모든 사람**이 그래프를 읽을 수 있다.
>
> 이것은 "배려"가 아니라 **전문성**이다. 학술 논문, 기업 보고서, AI 대시보드 모두
> 색맹 안전 팔레트를 사용하는 것이 표준이다.

### → AI 연결

- **AI 윤리**(AI Ethics): 공정한 AI는 모든 사용자를 고려해야 한다
- **접근성**(Accessibility): EU AI Act는 AI 시스템의 접근성을 법적으로 요구한다
- **의료 AI 대시보드**: 의사가 색맹이라면, 빨강/초록으로 구분된 위험도 표시를 읽지 못한다

### ✅ 3가지 습관

1. `palette="colorblind"` 또는 `cmap="viridis"` 사용
2. 색 + **형태**(마커, 선 스타일)를 함께 변경
3. 출력 전 `colorblindcheck.com`에서 시뮬레이션


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V11] 색맹 안전 팔레트 비교 — 기본 vs CVD-safe
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V11] 색맹 안전 팔레트 비교")
print(" ▶ 비유: 건물의 경사로(ramp)처럼, 색맹 안전 팔레트는")
print(" '모든 사람이 읽을 수 있는 그래프'를 만드는 유니버설 디자인이다.")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('기본 팔레트 vs 색맹 안전 팔레트 비교', fontsize=15, fontweight='bold')

np.random.seed(42)
categories = ['A', 'B', 'C', 'D', 'E']
values = [23, 45, 12, 67, 34]

# 위험한 팔레트 (빨강-초록)
danger_colors = ['#FF0000', '#00FF00', '#0000FF', '#FF00FF', '#00FFFF']
axes[0,0].bar(categories, values, color=danger_colors, edgecolor='black')
axes[0,0].set_title('❌ 위험: 빨강-초록 팔레트\n적녹 색맹이 구분 불가', fontweight='bold', color='red')

# 안전한 팔레트
axes[0,1].bar(categories, values, color=CVD_SAFE[:5], edgecolor='black')
axes[0,1].set_title('✅ 안전: CVD-safe 팔레트\n모든 사람이 구분 가능', fontweight='bold', color='green')

# 적녹 색맹 시뮬레이션 (Deuteranopia 근사)
def simulate_deuteranopia(hex_color):
 r = int(hex_color[1:3], 16) / 255
 g = int(hex_color[3:5], 16) / 255
 b = int(hex_color[5:7], 16) / 255
 # 간소화된 Deuteranopia 시뮬레이션
 new_r = 0.625 * r + 0.375 * g
 new_g = 0.70 * r + 0.30 * g
 new_b = 0.30 * g + 0.70 * b
 return (min(new_r,1), min(new_g,1), min(new_b,1))

danger_deut = [simulate_deuteranopia(c) for c in danger_colors]
axes[1,0].bar(categories, values, color=danger_deut, edgecolor='black')
axes[1,0].set_title('❌ 적녹 색맹이 본 위험 팔레트\n→ A와 B가 거의 같은 색!', fontweight='bold', color='red')

safe_hex = ['#0072B2', '#E69F00', '#009E73', '#CC79A7', '#56B4E9']
safe_deut = [simulate_deuteranopia(c) for c in safe_hex]
axes[1,1].bar(categories, values, color=safe_deut, edgecolor='black')
axes[1,1].set_title('✅ 적녹 색맹이 본 안전 팔레트\n→ 여전히 구분 가능!', fontweight='bold', color='green')

plt.tight_layout()
plt.show()

print()
print(" ▶ 핵심: 상단 왼쪽(빨강-초록)은 색맹에게 하단처럼 보인다.")
print(" → A와 B가 거의 같은 색이 되어 구분이 불가능하다!")
print(" ▶ 안전 팔레트는 색맹에게도 여전히 구분 가능하다.")
print()
print(" → AI 윤리: 공정한 AI는 '결과'만 공정한 것이 아니라,")
print(" '표현(시각화)'도 모든 사용자가 접근 가능해야 한다.")


---

## 1.6 그래프 선택 가이드 — "언제 무엇을 그리는가?"

### ▶ 비유: "**도구 상자**"

> 목수에게는 망치, 톱, 드릴, 줄자 등 다양한 도구가 있다.
> 못을 박을 때 톱을 쓰면 안 되듯, **데이터 유형에 맞는 그래프**를 선택해야 한다.

```
데이터 유형은?
├─ 수치형 1개 ──────→ 히스토그램 + KDE
├─ 수치형 2개 ──────→ 산점도(scatter)
├─ 수치형 여러 개 ──→ 상관 히트맵(heatmap)
├─ 범주 + 수치 ────→ 박스플롯 / 바이올린
├─ 범주 1개 ───────→ 카운트플롯(countplot)
└─ 시계열 ─────────→ 라인 플롯(lineplot)
```

| 목적 | Seaborn 함수 | 팔레트 지정 |
|---|---|---|
| 분포 확인 | `sns.histplot(kde=True)` | `color=CVD_SAFE[0]` |
| 그룹 비교 | `sns.boxplot()` | `palette="colorblind"` |
| 관계 탐색 | `sns.scatterplot()` | `palette="colorblind"` |
| 상관 행렬 | `sns.heatmap(cmap="viridis")` | `cmap="viridis"` |
| 범주 빈도 | `sns.countplot()` | `palette="colorblind"` |

### → AI 연결

- 데이터 유형 판별은 **자동 특성 공학**(AutoML)에서도 핵심 단계이다
- Auto-sklearn, H2O AutoML은 데이터 유형에 따라 자동으로 전처리를 결정한다


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-A2] 다중공선성(Multicollinearity) VIF 시각화
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "똑같은 말을 하는 두 친구"
# 시험 공부를 할 때, 친구 A가 "이 부분 중요해"라고 말하고
# 친구 B도 "이 부분 중요해"라고 똑같이 말하면?
# → 정보가 2배가 아니라, 사실상 같은 정보이다!
# → ML 모델도 같은 정보를 가진 두 특성이 있으면 혼란스러워한다.
# 이것이 "다중공선성"이다.

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

print("=" * 70)
print(" 다중공선성(Multicollinearity) — VIF 시각화")
print("=" * 70)
print()
print(" ▶ 비유: '똑같은 말을 하는 두 친구'")
print(" ─────────────────────────────────────────────")
print(" 친구 A: '이거 시험에 나와!'")
print(" 친구 B: '이거 시험에 나와!' (A가 한 말 그대로)")
print(" → 친구가 2명이지만, 새로운 정보는 0이다!")
print(" → ML에서 상관관계 r>0.9인 두 특성 = '똑같은 친구'")
print(" → 모델이 어떤 특성에 가중치를 줄지 혼란스러워한다!")
print()

np.random.seed(42)
n = 200

# 독립 특성
x1 = np.random.normal(50, 10, n)
x2 = x1 * 0.95 + np.random.normal(0, 3, n) # x1과 매우 상관
x3 = np.random.normal(30, 8, n) # 독립
x4 = x3 * 1.1 + np.random.normal(0, 2, n) # x3와 상관
x5 = np.random.normal(60, 12, n) # 독립

df_vif = pd.DataFrame({'키(cm)': x1, '팔길이(cm)': x2, 
 '공부시간(h)': x3, '과제횟수': x4, 
 '운동시간(h)': x5})

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='white')

# 1. 상관 행렬
corr = df_vif.corr()
im = axes[0].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(5))
axes[0].set_yticks(range(5))
labels = ['키', '팔길이', '공부시간', '과제횟수', '운동시간']
axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(labels, fontsize=9)
for i in range(5):
 for j in range(5):
 val = corr.iloc[i, j]
 color = 'white' if abs(val) > 0.5 else 'black'
 axes[0].text(j, i, f'{val:.2f}', ha='center', va='center', 
 fontsize=9, color=color, fontweight='bold')
axes[0].set_title('상관 행렬', fontweight='bold', fontsize=12)
plt.colorbar(im, ax=axes[0], fraction=0.046)

# 2. 다중공선성 쌍 (높은 상관)
axes[1].scatter(x1, x2, alpha=0.5, s=20, color='red')
axes[1].set_xlabel('키 (cm)')
axes[1].set_ylabel('팔길이 (cm)')
r_val = np.corrcoef(x1, x2)[0, 1]
axes[1].set_title(f'※ 높은 상관: r = {r_val:.3f}\n→ 다중공선성 위험!', 
 fontweight='bold', fontsize=11, color='red')
# 회귀선
z = np.polyfit(x1, x2, 1)
axes[1].plot(sorted(x1), np.polyval(z, sorted(x1)), 'r--', linewidth=2)

# 3. 독립 쌍 (낮은 상관)
axes[2].scatter(x1, x5, alpha=0.5, s=20, color='#2196F3')
axes[2].set_xlabel('키 (cm)')
axes[2].set_ylabel('운동시간 (h)')
r_val2 = np.corrcoef(x1, x5)[0, 1]
axes[2].set_title(f'✅ 낮은 상관: r = {r_val2:.3f}\n→ 독립적 정보!', 
 fontweight='bold', fontsize=11, color='green')
z2 = np.polyfit(x1, x5, 1)
axes[2].plot(sorted(x1), np.polyval(z2, sorted(x1)), 'b--', linewidth=2)

plt.suptitle('다중공선성(Multicollinearity): 같은 정보를 가진 특성 탐지', 
 fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print(" ▶ VIF(분산 팽창 인수) 기준:")
print(" ┌────────────┬──────────────────────────────┐")
print(" │ VIF < 5 │ ✅ 문제 없음 │")
print(" │ 5 < VIF │ ※ 주의 필요 │")
print(" │ VIF > 10 │ ❌ 심각! 특성 제거 고려 │")
print(" └────────────┴──────────────────────────────┘")
print()
print(" → AI 연결:")
print(" 딥러닝에서는 다중공선성이 덜 문제가 된다.")
print(" Dropout과 Regularization이 자동으로 중복 정보를 걸러내기 때문이다.")
print(" 하지만 해석 가능한 모델(선형 회귀, 로지스틱 회귀)에서는")
print(" VIF 점검이 필수이다!")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V12] 그래프 선택 가이드 — 6가지 그래프 유형 한눈에 보기
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V12] 6가지 그래프 유형 한눈에 보기")
print(" ▶ 비유: 목수의 도구 상자처럼, 상황에 맞는 그래프를 고르는 것이 핵심이다.")
print("=" * 60)

np.random.seed(42)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('그래프 선택 가이드 — "무엇을 알고 싶은가?"에 따라 그래프를 고른다',
 fontsize=14, fontweight='bold')

# 1. 히스토그램 + KDE
data = np.random.lognormal(4, 0.5, 500)
axes[0,0].hist(data, bins=30, density=True, color=CVD_SAFE[0], edgecolor='white', alpha=0.7)
from scipy.stats import gaussian_kde
kde_x = np.linspace(data.min(), data.max(), 200)
kde_y = gaussian_kde(data)(kde_x)
axes[0,0].plot(kde_x, kde_y, color='red', lw=2)
axes[0,0].set_title('① 히스토그램 + KDE\n"분포가 어떤 모양인가?"', fontweight='bold')

# 2. 박스플롯
groups = np.random.choice(['A', 'B', 'C'], 200)
vals = np.where(groups=='A', np.random.normal(50,10,200),
 np.where(groups=='B', np.random.normal(70,15,200), np.random.normal(60,8,200)))
temp_df = pd.DataFrame({'group': groups, 'value': vals})
temp_df.boxplot(column='value', by='group', ax=axes[0,1], patch_artist=True)
axes[0,1].set_title('② 박스플롯\n"그룹별 차이가 있는가?"', fontweight='bold')
axes[0,1].set_xlabel('')
plt.sca(axes[0,1])
plt.title('② 박스플롯\n"그룹별 차이가 있는가?"', fontweight='bold')

# 3. 산점도
x_sc = np.random.normal(0, 1, 100)
y_sc = 0.7*x_sc + np.random.normal(0, 0.5, 100)
axes[0,2].scatter(x_sc, y_sc, c=CVD_SAFE[2], alpha=0.6, edgecolors='black', linewidth=0.3)
z = np.polyfit(x_sc, y_sc, 1)
axes[0,2].plot(np.sort(x_sc), np.polyval(z, np.sort(x_sc)), 'r--', lw=2)
axes[0,2].set_title('③ 산점도 + 회귀선\n"두 변수의 관계는?"', fontweight='bold')

# 4. 히트맵
corr_mat = np.array([[1, 0.8, -0.3], [0.8, 1, -0.5], [-0.3, -0.5, 1]])
sns.heatmap(corr_mat, annot=True, fmt='.1f', cmap='viridis',
 xticklabels=['X1','X2','X3'], yticklabels=['X1','X2','X3'],
 ax=axes[1,0], linewidths=1, linecolor='white')
axes[1,0].set_title('④ 히트맵\n"전체 상관관계 한눈에"', fontweight='bold')

# 5. 카운트플롯
cats = np.random.choice(['서울', '부산', '대구', '인천'], 300, p=[0.4, 0.25, 0.2, 0.15])
cat_df = pd.DataFrame({'city': cats})
cat_df['city'].value_counts().plot.bar(ax=axes[1,1], color=CVD_SAFE[:4], edgecolor='black')
axes[1,1].set_title('⑤ 카운트플롯\n"범주별 빈도는?"', fontweight='bold')

# 6. 라인플롯
dates = pd.date_range('2024-01', periods=12, freq='M')
axes[1,2].plot(dates, np.cumsum(np.random.normal(5, 3, 12)), '-o', color=CVD_SAFE[0], lw=2)
axes[1,2].set_title('⑥ 라인 플롯\n"시간에 따른 변화는?"', fontweight='bold')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print()
print(" ▶ 그래프 선택 규칙:")
print(" • '분포?' → 히스토그램")
print(" • '그룹 비교?' → 박스플롯")
print(" • '관계?' → 산점도")
print(" • '전체 상관?' → 히트맵")
print(" • '빈도?' → 카운트플롯")
print(" • '추세?' → 라인 플롯")


---

## 1.7 EDA는 "선택"이 아니라 "필수" — ML **파이프라인**(Pipeline)의 관문

```
데이터 수집 → 【EDA】 → 전처리 → 특성 공학 → 모델 학습 → 평가 → 배포
```

### ▶ 비유: "**요리의 재료 검수**"

> 레스토랑에서 요리(모델 학습)를 시작하기 전에 반드시 **재료를 검수**한다.
>
> - 상한 재료 = **결측값·오류** → 제거하거나 대체한다
> - 크기가 다른 재료 = **스케일 차이** → 표준화한다
> - 알레르기 성분 = **데이터 누수** → 제거한다
>
> EDA 없이 모델을 학습하는 것은 **재료를 검수하지 않고 요리하는 것**이다.
> 맛없는 요리(나쁜 모델)가 나올 수밖에 없다.

### EDA에서 반드시 확인해야 하는 5가지

| # | 확인 항목 | 도구 | "왜?" |
|---|---|---|---|
| 1 | **형태·타입** | `df.shape`, `df.dtypes` | 특성 수, 데이터 타입 확인 |
| 2 | **결측값** | `df.isnull().sum()` | 결측 처리 전략 결정 |
| 3 | **분포** | 히스토그램, KDE | 정규성 위반 여부 확인 |
| 4 | **이상치** | 박스플롯, IQR | 오류 vs 희귀 사건 구분 |
| 5 | **상관관계** | 히트맵, 산점도 | 특성 선택·다중공선성 확인 |

### → AI 연결 — "데이터 중심 AI"(Data-Centric AI)

- Andrew Ng: **"80%의 시간을 데이터에, 20%를 모델에 써라"**
- 구글 PAIR: "데이터 품질이 모델 아키텍처보다 성능에 2~5배 더 큰 영향을 미친다"
- **MLOps 파이프라인**: 실제 프로덕션에서 EDA는 자동화된 데이터 모니터링으로 확장된다


---

# PART 2: 가이드 실습 — Seaborn·Plotly 핵심

> 이론에서 배운 **"왜?"**를 코드로 직접 확인한다. 모든 코드에 **색맹 안전 팔레트**가 적용되어 있다.

### ▶ 비유: "이론 = 교통법규, 실습 = 운전 연습"

> 아무리 교통법규(이론)를 많이 외워도, 실제로 핸들을 잡아보지 않으면 운전을 할 수 없다.
> PART 1에서 "왜 분포를 봐야 하는가?", "왜 이상치가 중요한가?"를 배웠다면,
> PART 2에서는 **실제 데이터**를 가지고 직접 그래프를 그려 본다.

| 그래프 | 목적 | 확인할 "왜?" |
|---|---|---|
| 히스토그램 + KDE | 분포 확인 | ML의 정규분포 가정 충족 여부 |
| 박스플롯 | 요약 + 이상치 | 이상치가 모델 성능에 미치는 영향 |
| 바이올린 플롯 | 분포 + 밀도 | 그룹 간 분포 차이 세부 확인 |
| 산점도 | 2변수 관계 | 특성 선택과 비선형 관계 탐지 |
| 히트맵 | 상관관계 | 다중공선성과 특성 중복 확인 |
| 페어플롯 | 모든 쌍 | 전체적 데이터 구조 파악 |
| 조인트플롯 | 관계+분포 | 관계와 개별 분포 동시 확인 |
| Plotly | 대화형 | 세밀한 탐색 (줌, 호버, 필터) |

> **→ AI 연결**: 실무 데이터 과학자는 모델 학습 **전에** 위 8가지 그래프를 반드시 그린다. 이 과정 없이 모델을 학습하면 **쓰레기를 넣으면 쓰레기가 나온다**(GIGO: Garbage In, Garbage Out)는 격언이 실현된다.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 데이터 로드 + 기본 전처리
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" Airbnb NYC 데이터 로드")
print("=" * 60)

url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/airbnb.csv"
try:
 df = pd.read_csv(url)
 print(f" ✅ 온라인 로드 성공: {df.shape[0]}행 × {df.shape[1]}열")
except:
 np.random.seed(42)
 n = 5000
 boroughs = np.random.choice(['Manhattan','Brooklyn','Queens','Bronx','Staten Island'],
 n, p=[0.44,0.30,0.15,0.06,0.05])
 df = pd.DataFrame({
 'neighbourhood_group': boroughs,
 'room_type': np.random.choice(['Entire home/apt','Private room','Shared room'], n, p=[0.52,0.45,0.03]),
 'price': np.random.lognormal(4.3, 0.8, n).astype(int),
 'minimum_nights': np.random.exponential(3, n).astype(int) + 1,
 'number_of_reviews': np.random.exponential(12, n).astype(int),
 'reviews_per_month': np.random.exponential(1.2, n).round(2),
 'latitude': np.random.normal(40.73, 0.05, n).round(5),
 'longitude': np.random.normal(-73.95, 0.04, n).round(5),
 })
 print(f" ※ 오프라인 대체 데이터 생성: {df.shape[0]}행 × {df.shape[1]}열")

# 이상치 필터링 (price ≤ 300)
print(f"\n 원본 데이터 기본 정보:")
print(f" • 행 수: {df.shape[0]:,}개")
print(f" • 열 수: {df.shape[1]}개")
print(f" • 가격 범위: ${df['price'].min():,} ~ ${df['price'].max():,}")
print(f" • 가격 평균: ${df['price'].mean():,.0f}")
print(f" • 가격 중앙값: ${df['price'].median():,.0f}")

df_under300 = df[df['price'].between(1, 300)].copy()
removed = len(df) - len(df_under300)
print(f"\n 가격 필터링 (1 ≤ price ≤ 300):")
print(f" • 제거: {removed}개 ({removed/len(df)*100:.1f}%)")
print(f" • 남은 데이터: {len(df_under300):,}개")
print()
print(" ▶ 비유: 건강검진에서 체온 500도처럼 불가능한 값은 먼저 제거한다.")
print(" 하지만 체온 39도(미열)는 제거하면 안 된다 — 이것은 의미 있는 이상치이다.")
print()
print(" → AI: 실제 Kaggle 대회에서도 첫 번째 단계는 항상 '가격 범위 확인'이다.")
print(" 극단값이 모델의 손실함수(Loss)를 왜곡하기 때문이다.")


---

## 2.1 **히스토그램**(Histogram) + KDE — 분포 확인

### ▶ 비유: "줄 세우기"

> 학생 100명을 키 순서대로 줄 세운 다음, 같은 키 구간끼리 묶으면 **히스토그램**이 된다.
> 그 묶음 위에 부드러운 곡선을 그리면 **KDE**(Kernel Density Estimation)이다.
>
> - **bins**: "몇 cm 단위로 묶을 것인가?" → bins가 크면 대략적, bins가 작으면 세밀하다
> - **KDE**: 울퉁불퉁한 히스토그램을 부드럽게 만든 것 → 확률 밀도의 추정치이다


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-W3] 인터랙티브: 그래프 유형 선택기 — "같은 데이터, 다른 관점"
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "같은 음식, 다른 접시"
# 스테이크를 종이접시에 담으면 → 분식집 느낌
# 스테이크를 도자기 접시에 담으면 → 레스토랑 느낌
# 같은 데이터도 "어떤 그래프에 담느냐"에 따라 인사이트가 달라진다!

import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 70)
print(" 인터랙티브: 같은 데이터를 6가지 그래프로 보기")
print("=" * 70)
print()
print(" ▶ 비유: 같은 풍경도 일출/일몰/야경으로 보면 느낌이 다르듯,")
print(" 같은 데이터도 그래프 유형에 따라 다른 인사이트가 보인다!")
print()

out_graph = widgets.Output()
graph_type = widgets.Dropdown(
 options=['① 히스토그램 (분포)', '② 박스플롯 (요약+이상치)', 
 '③ 바이올린 (분포+밀도)', '④ 스트립 (개별 점)',
 '⑤ 스웜 (겹침 없는 점)', '⑥ KDE (연속 밀도)'],
 value='① 히스토그램 (분포)', description='그래프:',
 style={'description_width': '60px'}, layout=widgets.Layout(width='350px')
)

# 샘플 데이터
np.random.seed(42)
cats = np.repeat(['A조', 'B조', 'C조'], 100)
vals = np.concatenate([
 np.random.normal(70, 10, 100),
 np.random.normal(80, 15, 100),
 np.random.normal(65, 8, 100)
])
import pandas as pd
df_demo = pd.DataFrame({'반': cats, '점수': vals})

def update_graph(*args):
 with out_graph:
 clear_output(wait=True)
 fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
 choice = graph_type.value[2:].split('(')[0].strip()
 
 insights = {
 '히스토그램': " 분포의 '모양'이 보인다: 봉우리 수, 편향, 범위",
 '박스플롯': " 5가지 요약 통계 + 이상치가 한눈에 보인다",
 '바이올린': " 박스플롯 + 밀도(분포 모양)를 동시에 볼 수 있다",
 '스트립': " 모든 개별 데이터 포인트의 위치가 보인다",
 '스웜': " 개별 점이 겹치지 않아서 데이터 밀도가 보인다",
 'KDE': " 연속적인 확률 밀도로 분포를 매끄럽게 본다"
 }
 
 if '히스토그램' in choice:
 sns.histplot(data=df_demo, x='점수', hue='반', palette='colorblind', 
 bins=25, alpha=0.6, ax=ax)
 elif '박스플롯' in choice:
 sns.boxplot(data=df_demo, x='반', y='점수', palette='colorblind', ax=ax)
 elif '바이올린' in choice:
 sns.violinplot(data=df_demo, x='반', y='점수', palette='colorblind', 
 inner='quart', ax=ax)
 elif '스트립' in choice:
 sns.stripplot(data=df_demo, x='반', y='점수', palette='colorblind', 
 alpha=0.5, jitter=True, ax=ax)
 elif '스웜' in choice:
 sns.swarmplot(data=df_demo, x='반', y='점수', palette='colorblind', 
 size=3, ax=ax)
 elif 'KDE' in choice:
 sns.kdeplot(data=df_demo, x='점수', hue='반', palette='colorblind', 
 fill=True, alpha=0.3, ax=ax)
 
 ax.set_title(f'{graph_type.value}', fontweight='bold', fontsize=13)
 plt.tight_layout()
 plt.show()
 
 print(f"\n {insights.get(choice, '')}")
 print(f" → 각 그래프는 서로 다른 질문에 답한다!")

graph_type.observe(update_graph, names='value')
display(widgets.VBox([graph_type, out_graph]))
update_graph()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V13] 히스토그램 + KDE — 평균 vs 중앙값
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V13] 히스토그램 + KDE")
print(" ▶ 비유: 학생 100명을 키 순서대로 줄 세운 것이 히스토그램이다.")
print(" 줄 위에 부드러운 곡선을 그리면 KDE이다.")
print("=" * 60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 전체 분포
mean_val = df_under300['price'].mean()
median_val = df_under300['price'].median()

sns.histplot(df_under300['price'], bins=50, kde=True, color=CVD_SAFE[0], ax=ax1, edgecolor='white')
ax1.axvline(mean_val, color='red', linestyle='--', lw=2, label=f'평균 = ${mean_val:.0f}')
ax1.axvline(median_val, color='green', linestyle='--', lw=2, label=f'중앙값 = ${median_val:.0f}')
ax1.set_title('전체 가격 분포 + KDE\n→ 오른쪽 꼬리: 평균 > 중앙값', fontweight='bold')
ax1.set_xlabel('가격 ($)')
ax1.set_ylabel('빈도')
ax1.legend()

# 로그 변환 후
log_prices = np.log1p(df_under300['price'])
log_mean = log_prices.mean()
log_median = log_prices.median()

sns.histplot(log_prices, bins=50, kde=True, color=CVD_SAFE[2], ax=ax2, edgecolor='white')
ax2.axvline(log_mean, color='red', linestyle='--', lw=2, label=f'평균 = {log_mean:.2f}')
ax2.axvline(log_median, color='green', linestyle='--', lw=2, label=f'중앙값 = {log_median:.2f}')
ax2.set_title('로그 변환 후 분포\n→ 정규분포에 가까워짐', fontweight='bold')
ax2.set_xlabel('log(가격+1)')
ax2.set_ylabel('빈도')
ax2.legend()

plt.tight_layout()
plt.show()

skew_orig = df_under300['price'].skew()
skew_log = log_prices.skew()
print(f"\n 비교:")
print(f" • 원본 왜도: {skew_orig:.2f} (오른쪽 꼬리가 길다)")
print(f" • 로그 변환 후 왜도: {skew_log:.2f} (정규분포에 가까움)")
print(f" • 평균과 중앙값 차이: ${mean_val - median_val:.0f} → 평균이 비싼 매물에 끌려간다")
print()
print(" ▶ '왜 로그 변환?': 대부분 $50~150이지만, 소수의 $300 매물이")
print(" 평균을 끌어올린다. 로그로 '압축'하면 ML 모델이 더 잘 학습한다.")
print()
print(" → AI: XGBoost, LightGBM 등 트리 기반 모델은 왜도에 강하지만,")
print(" 선형 회귀와 신경망은 로그 변환이 필수적이다.")


---

## 2.2 **박스플롯**(Box Plot) — 지역별 비교 + 이상치 확인

### ▶ 비유: "온도계"

> 박스플롯은 데이터의 "온도계"이다:
> - **상자**(Box): 데이터의 50%가 이 범위에 있다 (Q1~Q3)
> - **중앙선**: 중앙값 (반이 이보다 작고, 반이 이보다 크다)
> - **수염**(Whisker): 정상 범위의 끝 (Q1-1.5×IQR ~ Q3+1.5×IQR)
> - **점**(Flier): 수염 밖의 이상치 — 주의가 필요한 데이터


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V14] 박스플롯 구조(Anatomy) 시각적 설명
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V14] 박스플롯 구조 해부")
print(" ▶ 비유: 박스플롯은 데이터의 '온도계'이다.")
print(" 상자 안에 50%의 데이터가 있고, 수염 밖은 이상치이다.")
print("=" * 60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 왼쪽: 박스플롯 해부도
np.random.seed(42)
sample = np.concatenate([np.random.normal(100, 20, 200), [180, 190, 200, 10, 15]])
bp = ax1.boxplot(sample, vert=True, patch_artist=True, widths=0.4,
 boxprops=dict(facecolor=CVD_SAFE[4], alpha=0.6, linewidth=2),
 medianprops=dict(color='red', linewidth=2),
 flierprops=dict(marker='o', markerfacecolor=CVD_SAFE[5], markersize=8))

Q1 = np.percentile(sample, 25)
Q3 = np.percentile(sample, 75)
IQR = Q3 - Q1
med = np.median(sample)

# 주석 추가
ax1.annotate(f'Q3 = {Q3:.0f} (75%)', xy=(1.05, Q3), fontsize=11, fontweight='bold', color=CVD_SAFE[0],
 arrowprops=dict(arrowstyle='->', color=CVD_SAFE[0]))
ax1.annotate(f'중앙값 = {med:.0f} (50%)', xy=(1.05, med), fontsize=11, fontweight='bold', color='red',
 arrowprops=dict(arrowstyle='->', color='red'))
ax1.annotate(f'Q1 = {Q1:.0f} (25%)', xy=(1.05, Q1), fontsize=11, fontweight='bold', color=CVD_SAFE[0],
 arrowprops=dict(arrowstyle='->', color=CVD_SAFE[0]))
ax1.annotate(f'IQR = Q3-Q1 = {IQR:.0f}', xy=(0.6, (Q1+Q3)/2), fontsize=10,
 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax1.annotate(f'이상치!\n(수염 밖)', xy=(1, 195), fontsize=10, color='red', fontweight='bold',
 arrowprops=dict(arrowstyle='->', color='red'))
ax1.set_title('박스플롯 해부도 (Anatomy)', fontweight='bold', fontsize=13)
ax1.set_ylabel('값')

# 오른쪽: 지역별 비교
if 'neighbourhood_group' in df_under300.columns:
 order = df_under300.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).index
 sns.boxplot(x='neighbourhood_group', y='price', data=df_under300,
 order=order, palette='colorblind', ax=ax2)
 ax2.set_title('지역별 가격 비교\n→ Manhattan이 가장 비싸다', fontweight='bold', fontsize=13)
 ax2.set_xlabel('지역')
 ax2.set_ylabel('가격 ($)')
 ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print(f"\n 박스플롯 핵심 수치:")
print(f" • Q1 (25%) = {Q1:.0f}")
print(f" • 중앙값 (50%) = {med:.0f}")
print(f" • Q3 (75%) = {Q3:.0f}")
print(f" • IQR = Q3 - Q1 = {IQR:.0f}")
print(f" • 이상치 경계: [{Q1-1.5*IQR:.0f}, {Q3+1.5*IQR:.0f}]")
print()
print(" → AI: 특성 공학에서 IQR 기반 이상치 제거는 가장 기본적인 전처리이다.")
print(" Kaggle 상위 솔루션의 95%가 이상치 처리를 포함한다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V15] 바이올린 플롯 — 박스플롯 + 분포 형태
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V15] 바이올린 플롯 = 박스플롯 + KDE 결합")
print(" ▶ 비유: 박스플롯이 '요약표'라면, 바이올린 플롯은 '풀버전'이다.")
print(" 분포의 모양(이봉분포 등)까지 보여 준다.")
print("=" * 60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

if 'neighbourhood_group' in df_under300.columns and 'room_type' in df_under300.columns:
 top3 = df_under300['neighbourhood_group'].value_counts().head(3).index
 sub = df_under300[df_under300['neighbourhood_group'].isin(top3)]

 sns.violinplot(x='neighbourhood_group', y='price', data=sub,
 palette='colorblind', ax=ax1, inner='box', cut=0)
 ax1.set_title('지역별 가격 바이올린 플롯\n→ 분포 형태까지 보인다', fontweight='bold')
 ax1.set_xlabel('지역')
 ax1.set_ylabel('가격 ($)')

 sns.violinplot(x='room_type', y='price', data=sub,
 palette='colorblind', ax=ax2, inner='quartile', cut=0)
 ax2.set_title('숙소 유형별 가격\n→ Entire home이 넓고 높다', fontweight='bold')
 ax2.set_xlabel('숙소 유형')
 ax2.set_ylabel('가격 ($)')
 ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print()
print(" ▶ 바이올린 플롯 읽기:")
print(" • 폭이 넓은 곳 = 데이터가 많이 몰려 있는 구간")
print(" • 폭이 좁은 곳 = 데이터가 적은 구간")
print(" • 이봉분포(봉우리 2개)가 보이면 → 2개 그룹이 섞여 있다는 증거")
print()
print(" → AI: GAN의 mode collapse 진단에도 분포 시각화를 사용한다.")
print(" 생성된 데이터의 분포가 실제 데이터와 다르면 → mode collapse이다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V16] 산점도 — 위치 × 가격 (NYC 지도 느낌)
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V16] 산점도 — 위치별 가격 분포")
print(" ▶ 비유: 부동산 앱에서 지도에 점을 찍어 가격대를 색으로 표시한 것이다.")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 8))

if 'latitude' in df_under300.columns and 'longitude' in df_under300.columns:
 sample = df_under300.sample(min(2000, len(df_under300)), random_state=42)
 scatter = ax.scatter(sample['longitude'], sample['latitude'],
 c=sample['price'], cmap='viridis', s=10, alpha=0.5)
 plt.colorbar(scatter, ax=ax, label='가격 ($)')
 ax.set_title('NYC Airbnb — 위치별 가격 분포\n→ 색이 밝을수록 비싸다 (viridis: 색맹 안전)',
 fontweight='bold', fontsize=13)
 ax.set_xlabel('경도 (Longitude)')
 ax.set_ylabel('위도 (Latitude)')

plt.tight_layout()
plt.show()

print()
print(" ▶ 관찰: Manhattan(중앙)이 노란색(비싸다), 외곽은 보라색(저렴하다)")
print(" ▶ viridis 컬러맵은 색맹에게도 밝기 차이로 구분 가능하다.")
print()
print(" → AI: 추천 시스템에서 '위치'는 강력한 특성(feature)이다.")
print(" Airbnb의 실제 ML 모델도 위도·경도를 특성으로 사용한다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V17] 상관 히트맵 — 삼각형 + 수치 표시
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V17] 상관 히트맵")
print(" ▶ 비유: '친구 관계도'처럼 모든 변수 쌍의 친밀도를 한 장에 본다.")
print(" 상삼각은 하삼각의 거울이므로, 한쪽만 보면 충분하다.")
print("=" * 60)

numeric_cols = df_under300.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) > 2:
 corr = df_under300[numeric_cols].corr()
 mask = np.triu(np.ones_like(corr, dtype=bool))

 fig, ax = plt.subplots(figsize=(10, 8))
 sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='viridis',
 center=0, linewidths=1, linecolor='white', ax=ax,
 vmin=-1, vmax=1,
 cbar_kws={'label': '상관계수 (Pearson r)'})
 ax.set_title('수치형 변수 상관 히트맵 (하삼각)\n→ |r| > 0.7이면 다중공선성 의심',
 fontweight='bold', fontsize=13)

 plt.tight_layout()
 plt.show()

 # 강한 상관관계 찾기
 print()
 print(" 주요 상관관계 (|r| > 0.3):")
 for i in range(len(corr)):
 for j in range(i+1, len(corr)):
 r_val = corr.iloc[i, j]
 if abs(r_val) > 0.3:
 print(f" • {corr.index[i]} ↔ {corr.columns[j]}: r = {r_val:.3f}")

print()
print(" → AI: 다중공선성(|r| > 0.9)이 있으면 선형 회귀의 계수가 불안정해진다.")
print(" → VIF(분산팽창인자) 또는 PCA로 해결한다.")


---

## 2.3 **Plotly** — 대화형 그래프

### ▶ 비유: "**Google Earth** vs 종이 지도"

> Seaborn(정적 그래프) = 종이 지도: 한번 인쇄하면 변경 불가
> Plotly(대화형 그래프) = Google Earth: 확대·축소·회전·클릭 가능
>
> 대화형 그래프는 **탐색**(Exploration) 단계에서 특히 유용하다.
> 마우스를 올리면 정확한 값을 확인할 수 있고, 영역을 확대할 수 있다.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V18] Plotly 대화형 산점도 + 박스플롯
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V18] Plotly 대화형 그래프")
print(" ▶ 비유: 종이 지도(Seaborn) vs Google Earth(Plotly)")
print(" → 확대, 축소, 호버(마우스 올리기)가 가능하다.")
print("=" * 60)

try:
 import plotly.express as px
 import plotly.graph_objects as go

 sample = df_under300.sample(min(1000, len(df_under300)), random_state=42)

 # 대화형 산점도
 if 'neighbourhood_group' in sample.columns:
 fig1 = px.scatter(sample, x='longitude', y='latitude', color='neighbourhood_group',
 size='price', hover_data=['price', 'room_type'],
 title='[V18a] NYC Airbnb 대화형 지도 — 마우스를 올려 보라',
 color_discrete_sequence=px.colors.qualitative.Safe)
 fig1.show()

 # 대화형 박스플롯
 fig2 = px.box(sample, x='neighbourhood_group', y='price', color='room_type',
 title='[V18b] 지역 × 숙소유형별 가격 — 클릭하여 필터링 가능',
 color_discrete_sequence=px.colors.qualitative.Safe)
 fig2.show()
 print(" ✅ Plotly 대화형 그래프 2개 생성")
 else:
 print(" ※ neighbourhood_group 열이 없어 대체 그래프를 생성한다.")
except ImportError:
 print(" ※ Plotly가 설치되지 않았다. 'pip install plotly'로 설치하라.")
 print(" Plotly는 대화형 그래프 라이브러리로, 확대·축소·호버 기능을 제공한다.")

print()
print(" → AI: TensorBoard(딥러닝 시각화 도구)도 Plotly와 유사한")
print(" 대화형 인터페이스로 학습 곡선, 히스토그램, 임베딩을 시각화한다.")


---

## 2.4 인터랙티브 위젯 — 직접 조작하며 배우기

### ▶ 비유: "과학 실험실"

> 이론 수업에서 "bins를 바꾸면 분포가 달라진다"고 들었다.
> 하지만 **직접 슬라이더를 돌려보면** 그 의미를 체감할 수 있다.
> 이것은 화학 실험에서 시약 양을 바꿔 보는 것과 같다.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-A3] 페어플롯(Pairplot) — "모든 특성 쌍을 한눈에"
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "소개팅 사진"
# 소개팅 전에 사진 한 장만 보면 한 면만 안다.
# 여러 각도의 사진을 보면 전체적인 인상을 알 수 있다.
# 페어플롯 = 데이터의 "모든 각도 사진"이다!

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("=" * 70)
print(" 페어플롯(Pairplot) — 모든 특성 쌍의 관계를 한눈에")
print("=" * 70)
print()
print(" ▶ 비유: 소개팅 사진")
print(" ─────────────────────────────────────")
print(" 정면 사진 1장 = 산점도 1개 (부분적)")
print(" 모든 각도 사진 = 페어플롯 (전체적)")
print(" → 데이터의 '전체 인상'을 파악할 수 있다!")
print()
print(" 대각선: 각 특성의 분포 (히스토그램/KDE)")
print(" 비대각선: 두 특성 간의 산점도")
print(" 색상: hue 변수로 그룹 구분")
print()

# 간단한 데이터로 페어플롯
from sklearn.datasets import load_iris
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=['꽃받침 길이', '꽃받침 너비', '꽃잎 길이', '꽃잎 너비'])
df_iris['품종'] = [iris.target_names[t] for t in iris.target]

g = sns.pairplot(df_iris, hue='품종', palette='colorblind', 
 diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20},
 height=2.5)
g.figure.suptitle('Iris 데이터 페어플롯 — 4개 특성 × 3개 품종', y=1.02, fontweight='bold')
plt.show()

print()
print(" ▶ 관찰 포인트:")
print(" 1. 꽃잎 길이 vs 꽃잎 너비 → 품종 분리가 가장 뚜렷하다")
print(" 2. 꽃받침 너비만으로는 → 3종 구분이 어렵다")
print(" 3. setosa는 → 다른 품종과 확실히 분리된다")
print()
print(" → AI 연결:")
print(" 페어플롯은 특성 공학(Feature Engineering)의 출발점이다.")
print(" 어떤 특성 쌍이 분류에 유용한지 시각적으로 판단하고,")
print(" 이를 바탕으로 ML 모델의 입력 특성을 선택한다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V19] 인터랙티브 위젯 ① — bins 수 + 팔레트 전환
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V19] 인터랙티브 — bins와 팔레트를 슬라이더로 조작")
print(" ▶ 비유: 현미경의 배율 조절처럼, bins를 바꾸면")
print(" 데이터의 다른 면이 보인다.")
print("=" * 60)

import ipywidgets as widgets
from IPython.display import display, clear_output

def plot_histogram(bins=30, palette='colorblind', show_kde=True, show_stats=True):
 fig, ax = plt.subplots(figsize=(10, 5))
 color = CVD_SAFE[0] if palette == 'colorblind' else 'steelblue'
 sns.histplot(df_under300['price'], bins=bins, kde=show_kde, color=color, ax=ax, edgecolor='white')

 if show_stats:
 ax.axvline(df_under300['price'].mean(), color='red', linestyle='--', lw=2, label=f"평균=${df_under300['price'].mean():.0f}")
 ax.axvline(df_under300['price'].median(), color='green', linestyle='--', lw=2, label=f"중앙값=${df_under300['price'].median():.0f}")
 ax.legend(fontsize=11)

 ax.set_title(f'가격 분포 (bins={bins}, palette={palette})', fontweight='bold', fontsize=13)
 ax.set_xlabel('가격 ($)')
 ax.set_ylabel('빈도')
 plt.tight_layout()
 plt.show()

 print(f" → bins={bins}: ", end="")
 if bins < 15:
 print("큰 덩어리로 보인다 (디테일 부족)")
 elif bins < 40:
 print("적절한 수준의 디테일")
 else:
 print("매우 세밀하다 (과적합 위험과 비슷한 개념!)")

try:
 widgets.interact(
 plot_histogram,
 bins=widgets.IntSlider(min=5, max=100, step=5, value=30, description='Bins:'),
 palette=widgets.Dropdown(options=['colorblind', 'default'], value='colorblind', description='팔레트:'),
 show_kde=widgets.Checkbox(value=True, description='KDE 표시'),
 show_stats=widgets.Checkbox(value=True, description='통계량 표시')
 )
except:
 plot_histogram(30)
 print(" ※ 위젯이 지원되지 않는 환경이다. 기본값으로 표시하였다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-A4] 조인트플롯(Jointplot) — "두 변수의 관계 + 개별 분포"
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "X선 + MRI 동시 촬영"
# X선 = 두 변수의 관계(산점도)
# MRI = 각 변수의 세부 분포(히스토그램)
# 조인트플롯 = 두 검사를 한 화면에!

print("=" * 70)
print(" 조인트플롯(Jointplot) — 관계 + 분포를 한 번에")
print("=" * 70)
print()
print(" ▶ 비유: 'X선 + MRI 동시 촬영'")
print(" 중앙 = 두 변수의 관계(산점도)")
print(" 위/오른쪽 = 각 변수의 분포(히스토그램)")
print()

fig = sns.jointplot(data=df_under300.sample(2000, random_state=42), 
 x='number_of_reviews', y='price',
 hue='room_type', palette='colorblind',
 kind='scatter', alpha=0.4, height=7)
fig.figure.suptitle('리뷰 수 vs 가격 — room_type별', y=1.02, fontweight='bold')
plt.show()

print()
print(" ▶ 관찰:")
print(" 1. 리뷰가 많은 숙소 → 대체로 저가(접근성 높은 가격)")
print(" 2. 고가 숙소 → 리뷰가 적다(희소한 고급 시장)")
print(" 3. Entire home/apt → 가격 범위가 넓다")
print()
print(" → AI 연결:")
print(" 추천 시스템(Netflix, Amazon)은 사용자-아이템 간 관계를")
print(" 이와 유사한 2D 공간에서 분석한다. 리뷰 수와 가격의 관계는")
print(" '인기도 vs 프리미엄' 축으로 해석할 수 있다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V20] 인터랙티브 위젯 ② — 상관계수 강도 체험
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V20] 인터랙티브 — 상관계수 r을 바꿔 보라")
print(" ▶ 비유: 고무줄의 탄성처럼, r이 1에 가까우면 점들이")
print(" 직선 위에 꽉 모이고, r이 0이면 흩어진다.")
print("=" * 60)

def plot_correlation(r=0.8, n_points=200, noise_type='normal'):
 np.random.seed(42)
 x = np.random.normal(0, 1, n_points)
 if noise_type == 'normal':
 noise = np.random.normal(0, np.sqrt(1 - r**2 + 0.01), n_points)
 else:
 noise = np.random.uniform(-np.sqrt(3*(1-r**2+0.01)), np.sqrt(3*(1-r**2+0.01)), n_points)
 y = r * x + noise

 actual_r = np.corrcoef(x, y)[0, 1]

 fig, ax = plt.subplots(figsize=(8, 6))
 ax.scatter(x, y, c=CVD_SAFE[0], alpha=0.5, edgecolors='black', linewidth=0.3, s=30)
 z = np.polyfit(x, y, 1)
 ax.plot(np.sort(x), np.polyval(z, np.sort(x)), 'r--', lw=2)
 ax.set_title(f'설정 r = {r:.2f}, 실제 r = {actual_r:.3f}', fontweight='bold', fontsize=14)
 ax.set_xlabel('X')
 ax.set_ylabel('Y')
 ax.set_xlim(-4, 4)
 ax.set_ylim(-4, 4)
 ax.grid(True, alpha=0.3)
 plt.tight_layout()
 plt.show()

 if abs(actual_r) > 0.7:
 print(f" → |r| = {abs(actual_r):.2f} > 0.7: 강한 상관 — 점들이 직선에 가깝다")
 elif abs(actual_r) > 0.3:
 print(f" → |r| = {abs(actual_r):.2f}: 중간 상관 — 경향은 보이지만 흩어져 있다")
 else:
 print(f" → |r| = {abs(actual_r):.2f} < 0.3: 약한 상관 — 거의 랜덤처럼 보인다")

try:
 widgets.interact(
 plot_correlation,
 r=widgets.FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.8, description='r:'),
 n_points=widgets.IntSlider(min=50, max=500, step=50, value=200, description='데이터 수:'),
 noise_type=widgets.Dropdown(options=['normal', 'uniform'], description='노이즈:')
 )
except:
 plot_correlation(0.8)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V21] 인터랙티브 위젯 ③ — 분포 유형 탐색기
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V21] 분포 유형 탐색기")
print(" ▶ 비유: '시험 난이도 조절기'를 돌려보며")
print(" 분포가 어떻게 변하는지 직접 체감한다.")
print("=" * 60)

def explore_distribution(dist_type='정규분포', sample_size=1000, param=1.0):
 np.random.seed(42)
 if dist_type == '정규분포':
 data = np.random.normal(50, param * 10, sample_size)
 desc = f"평균=50, 표준편차={param*10:.0f}"
 elif dist_type == '지수분포':
 data = np.random.exponential(param * 10, sample_size)
 desc = f"λ={1/(param*10):.2f}"
 elif dist_type == '균등분포':
 data = np.random.uniform(0, param * 100, sample_size)
 desc = f"범위=[0, {param*100:.0f}]"
 elif dist_type == '이봉분포':
 data = np.concatenate([
 np.random.normal(30, 5, sample_size//2),
 np.random.normal(30 + param * 40, 5, sample_size//2)
 ])
 desc = f"봉우리 간격={param*40:.0f}"

 fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
 ax1.hist(data, bins=40, color=CVD_SAFE[0], edgecolor='white', alpha=0.8, density=True)
 ax1.set_title(f'{dist_type} ({desc})', fontweight='bold')
 ax1.set_ylabel('확률 밀도')

 stats.probplot(data, dist="norm", plot=ax2)
 ax2.set_title('QQ-plot (정규성 검정)', fontweight='bold')
 ax2.get_lines()[0].set_color(CVD_SAFE[0])

 plt.tight_layout()
 plt.show()

 skewness = stats.skew(data)
 kurtosis_val = stats.kurtosis(data)
 print(f" 왜도(Skewness) = {skewness:.3f}, 첨도(Kurtosis) = {kurtosis_val:.3f}")
 if abs(skewness) < 0.5:
 print(" → 대칭에 가깝다 (ML 알고리즘에 적합)")
 else:
 print(" → 비대칭이다 (변환 고려 필요)")

try:
 widgets.interact(
 explore_distribution,
 dist_type=widgets.Dropdown(options=['정규분포', '지수분포', '균등분포', '이봉분포'], description='분포:'),
 sample_size=widgets.IntSlider(min=100, max=5000, step=100, value=1000, description='표본 수:'),
 param=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='파라미터:')
 )
except:
 explore_distribution('정규분포')


---

# PART 3: 프로젝트 — "어디가 비싸고, 왜 비싼가?"

> **미션**: 에어비앤비 NYC 데이터를 시각적으로 탐색하여, **수치를 포함한 관찰**을 최소 3개 도출하라.

### ▶ 비유: "**탐정의 수사 보고서**"

> 탐정(데이터 과학자)이 사건 현장(데이터)을 조사한다.
> - 증거 1번(분포): "피해자의 90%가 $50~$150 범위에 있다"
> - 증거 2번(비교): "Manhattan의 중앙값($150)은 Bronx($65)의 2.3배이다"
> - 증거 3번(관계): "리뷰 수와 가격의 상관계수는 -0.15로 약한 음의 상관이다"
>
> **나쁜 관찰**: "Manhattan이 비싸다" ← 수치 없음
> **좋은 관찰**: "Manhattan 중앙값 $150은 전체 중앙값 $95의 1.6배이다" ← 수치 포함

### → AI 연결

- 이 6단계 EDA 파이프라인은 **Kaggle 대회의 표준 접근법**이다
- 실제 AI 프로젝트의 80%는 데이터 탐색·전처리에 소요된다
- **MLOps**: 프로덕션 환경에서 이 파이프라인은 자동화된 모니터링으로 확장된다


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V22] 1단계: 전체 가격 분포 + 기술 통계
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V22] 1단계: 전체 가격 분포")
print(" ▶ 비유: 수사의 첫 단계는 '전체 현장 파악'이다.")
print("=" * 60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램 + KDE
sns.histplot(df_under300['price'], bins=50, kde=True, color=CVD_SAFE[0], ax=ax1, edgecolor='white')
ax1.axvline(df_under300['price'].mean(), color='red', linestyle='--', lw=2, label=f"평균=${df_under300['price'].mean():.0f}")
ax1.axvline(df_under300['price'].median(), color='green', linestyle='--', lw=2, label=f"중앙값=${df_under300['price'].median():.0f}")
ax1.set_title('1단계: 전체 가격 분포', fontweight='bold', fontsize=13)
ax1.set_xlabel('가격 ($)')
ax1.legend()

# CDF
sorted_prices = np.sort(df_under300['price'])
cdf = np.arange(1, len(sorted_prices)+1) / len(sorted_prices)
ax2.plot(sorted_prices, cdf, color=CVD_SAFE[0], lw=2)
ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax2.axhline(0.9, color='gray', linestyle=':', alpha=0.5)
ax2.set_title('누적분포함수(CDF)\n→ 50%가 중앙값 이하, 90%가 어디?', fontweight='bold')
ax2.set_xlabel('가격 ($)')
ax2.set_ylabel('누적 비율')

# 90% 지점 표시
p90 = np.percentile(df_under300['price'], 90)
ax2.axvline(p90, color='red', linestyle='--', label=f'90% = ${p90:.0f}')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n 관찰 1: 가격 분포는 오른쪽 꼬리 형태이다")
print(f" • 평균(${df_under300['price'].mean():.0f}) > 중앙값(${df_under300['price'].median():.0f}) → 양의 왜도")
print(f" • 90%의 매물이 ${p90:.0f} 이하에 분포한다")
print(f" • 왜도 = {df_under300['price'].skew():.2f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V23] 2단계: 지역별 비교 — 박스플롯 + 바이올린
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V23] 2단계: 지역별 가격 비교")
print(" ▶ 비유: 서울도 강남과 강북의 집값이 다르듯, NYC도 지역마다 다르다.")
print("=" * 60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if 'neighbourhood_group' in df_under300.columns:
 order = df_under300.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False).index

 sns.boxplot(x='neighbourhood_group', y='price', data=df_under300,
 order=order, palette='colorblind', ax=ax1)
 ax1.set_title('지역별 가격 박스플롯', fontweight='bold')
 ax1.tick_params(axis='x', rotation=15)

 sns.violinplot(x='neighbourhood_group', y='price', data=df_under300,
 order=order, palette='colorblind', ax=ax2, inner='quartile', cut=0)
 ax2.set_title('지역별 가격 바이올린 플롯', fontweight='bold')
 ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

if 'neighbourhood_group' in df_under300.columns:
 print("\n 관찰 2: 지역별 가격 중앙값 비교")
 medians = df_under300.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False)
 for region, val in medians.items():
 ratio = val / medians.min()
 print(f" • {region}: ${val:.0f} (최저가 대비 {ratio:.1f}배)")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V24] 3단계: 위치 × 가격 산점도 + 숙소유형별 분리
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V24] 3단계: 위치별 가격 산점도")
print(" ▶ 비유: 부동산 앱의 지도 뷰처럼 '어디가 비싼지' 시각적으로 파악한다.")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if all(c in df_under300.columns for c in ['latitude', 'longitude', 'room_type']):
 sample = df_under300.sample(min(2000, len(df_under300)), random_state=42)

 # 가격별 색상
 sc = axes[0].scatter(sample['longitude'], sample['latitude'],
 c=sample['price'], cmap='viridis', s=8, alpha=0.5)
 plt.colorbar(sc, ax=axes[0], label='가격 ($)')
 axes[0].set_title('가격별 색상', fontweight='bold')

 # 숙소 유형별 색상
 for i, rt in enumerate(sample['room_type'].unique()):
 sub = sample[sample['room_type'] == rt]
 axes[1].scatter(sub['longitude'], sub['latitude'],
 c=CVD_SAFE[i], s=8, alpha=0.5, label=rt)
 axes[1].legend(markerscale=3)
 axes[1].set_title('숙소 유형별 색상', fontweight='bold')

 for ax in axes:
 ax.set_xlabel('경도')
 ax.set_ylabel('위도')

plt.tight_layout()
plt.show()

print()
print(" 관찰 3: 위치와 가격의 관계")
print(" • Manhattan(중앙): 밝은 색(고가) 집중")
print(" • Entire home/apt: 주로 Manhattan에 집중")
print(" • Shared room: 외곽 지역에 분산")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V25] 4단계: room_type × 지역 교차 히트맵
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V25] 4단계: 교차 히트맵")
print(" ▶ 비유: 엑셀의 피벗 테이블을 색으로 표현한 것이다.")
print(" 행(지역) × 열(숙소유형) 조합별 가격을 한눈에 본다.")
print("=" * 60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if all(c in df_under300.columns for c in ['neighbourhood_group', 'room_type']):
 # 평균 가격 피벗
 pivot_price = df_under300.pivot_table(values='price', index='neighbourhood_group',
 columns='room_type', aggfunc='median')
 sns.heatmap(pivot_price, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax1,
 linewidths=1, linecolor='white')
 ax1.set_title('지역 × 숙소유형 중앙 가격', fontweight='bold')

 # 매물 수 피벗
 pivot_count = df_under300.pivot_table(values='price', index='neighbourhood_group',
 columns='room_type', aggfunc='count')
 sns.heatmap(pivot_count, annot=True, fmt='.0f', cmap='viridis', ax=ax2,
 linewidths=1, linecolor='white')
 ax2.set_title('지역 × 숙소유형 매물 수', fontweight='bold')

plt.tight_layout()
plt.show()

print()
print(" 관찰 4: 교차 분석 인사이트")
print(" • Manhattan Entire home이 가장 비싸고 매물도 많다")
print(" • Shared room은 모든 지역에서 가장 저렴하다")
print()
print(" → AI: 이러한 교차 분석은 '교차 특성(Interaction Feature)'의 기초이다.")
print(" 예: region_roomtype = Manhattan_Entire → 하나의 범주형 특성으로 사용")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V26] 5단계: 상관 분석 + 페어플롯
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V26] 5단계: 상관 분석")
print(" ▶ 비유: 모든 변수의 '관계도'를 한 장에 그린다.")
print("=" * 60)

numeric_cols = df_under300.select_dtypes(include=[np.number]).columns.tolist()
key_cols = [c for c in ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month'] if c in numeric_cols]

if len(key_cols) >= 2:
 fig, ax = plt.subplots(figsize=(8, 6))
 corr = df_under300[key_cols].corr()
 mask = np.triu(np.ones_like(corr, dtype=bool))
 sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='viridis',
 center=0, ax=ax, linewidths=1, linecolor='white', vmin=-1, vmax=1)
 ax.set_title('핵심 변수 상관 히트맵', fontweight='bold', fontsize=13)
 plt.tight_layout()
 plt.show()

 print()
 print(" 관찰 5: 상관관계 해석")
 for i in range(len(corr)):
 for j in range(i+1, len(corr)):
 r_val = corr.iloc[i, j]
 strength = "강한" if abs(r_val)>0.5 else "중간" if abs(r_val)>0.3 else "약한"
 direction = "양의" if r_val > 0 else "음의"
 print(f" • {corr.index[i]} ↔ {corr.columns[j]}: r={r_val:.3f} ({strength} {direction} 상관)")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V27] 6단계: 종합 관찰 + 대시보드
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V27] 6단계: 종합 EDA 대시보드")
print(" ▶ 비유: 탐정이 수사 보고서를 작성하듯,")
print(" EDA 결과를 한 장에 요약한다.")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('NYC Airbnb EDA 종합 대시보드', fontsize=16, fontweight='bold')

# 1. 가격 분포
sns.histplot(df_under300['price'], bins=40, kde=True, color=CVD_SAFE[0], ax=axes[0,0], edgecolor='white')
axes[0,0].set_title('① 가격 분포', fontweight='bold')

# 2. 지역별 박스플롯
if 'neighbourhood_group' in df_under300.columns:
 sns.boxplot(x='neighbourhood_group', y='price', data=df_under300, palette='colorblind', ax=axes[0,1])
 axes[0,1].tick_params(axis='x', rotation=30)
axes[0,1].set_title('② 지역별 비교', fontweight='bold')

# 3. 숙소 유형별
if 'room_type' in df_under300.columns:
 df_under300['room_type'].value_counts().plot.bar(color=CVD_SAFE[:3], ax=axes[0,2], edgecolor='black')
axes[0,2].set_title('③ 숙소 유형 빈도', fontweight='bold')

# 4. 위치 산점도
if 'latitude' in df_under300.columns:
 s = df_under300.sample(min(1000, len(df_under300)), random_state=42)
 axes[1,0].scatter(s['longitude'], s['latitude'], c=s['price'], cmap='viridis', s=5, alpha=0.5)
axes[1,0].set_title('④ 위치 × 가격', fontweight='bold')

# 5. 상관 히트맵
if len(key_cols) >= 2:
 sns.heatmap(df_under300[key_cols].corr(), annot=True, fmt='.2f', cmap='viridis',
 ax=axes[1,1], linewidths=0.5, vmin=-1, vmax=1, center=0)
axes[1,1].set_title('⑤ 상관 히트맵', fontweight='bold')

# 6. 가격대별 비율
bins_price = [0, 50, 100, 150, 200, 300]
labels = ['~$50', '$51-100', '$101-150', '$151-200', '$201-300']
df_under300['price_range'] = pd.cut(df_under300['price'], bins=bins_price, labels=labels)
df_under300['price_range'].value_counts().sort_index().plot.barh(
 color=CVD_SAFE[:5], ax=axes[1,2], edgecolor='black')
axes[1,2].set_title('⑥ 가격대별 매물 수', fontweight='bold')

plt.tight_layout()
plt.show()

print()
print(" ▶ 종합 관찰 보고서:")
print(" ─" * 30)
n_total = len(df_under300)
print(f" 1. 총 {n_total:,}개 매물 중 90%가 ${np.percentile(df_under300['price'], 90):.0f} 이하이다.")
if 'neighbourhood_group' in df_under300.columns:
 top = df_under300.groupby('neighbourhood_group')['price'].median().idxmax()
 top_val = df_under300.groupby('neighbourhood_group')['price'].median().max()
 print(f" 2. {top}의 중앙 가격 ${top_val:.0f}이 가장 비싸다.")
if 'room_type' in df_under300.columns:
 entire_pct = (df_under300['room_type'] == 'Entire home/apt').mean() * 100
 print(f" 3. Entire home/apt 비율이 {entire_pct:.1f}%로 가장 높다.")
print()
print(" → AI: 이 6단계 EDA 파이프라인은 Kaggle 상위 10% 솔루션의 표준 접근법이다.")
print(" 실제 AI 프로젝트에서도 동일한 순서로 데이터를 탐색한다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-W4] 인터랙티브: FacetGrid — 조건별 소화면 자동 생성
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "CCTV 모니터 화면"
# 보안실에서 CCTV 16분할 화면을 보면 → 모든 구역을 한눈에!
# FacetGrid = 데이터의 "CCTV 모니터"이다.
# 각 조건별 소화면을 자동으로 생성하여 비교한다.

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 70)
print(" 인터랙티브: FacetGrid — 조건별 소그래프 생성")
print("=" * 70)
print()
print(" ▶ 비유: CCTV 모니터")
print(" ─────────────────────────────────────────")
print(" 1개 CCTV = 산점도 1개 (한 구역만)")
print(" 16분할 모니터 = FacetGrid (모든 구역을 동시에)")
print(" → 지역별·유형별 패턴을 한눈에 비교할 수 있다!")
print()

out_facet = widgets.Output()
col_var = widgets.Dropdown(
 options=['neighbourhood_group', 'room_type'],
 value='neighbourhood_group', description='열 기준:',
 style={'description_width': '70px'})
graph_var = widgets.Dropdown(
 options=['히스토그램', '산점도', 'KDE'],
 value='히스토그램', description='그래프:',
 style={'description_width': '70px'})

def update_facet(*args):
 with out_facet:
 clear_output(wait=True)
 col = col_var.value
 
 if '히스토그램' in graph_var.value:
 g = sns.FacetGrid(df_under300, col=col, col_wrap=3, height=3.5,
 sharex=True, sharey=True)
 g.map(sns.histplot, 'price', bins=30, color='#2196F3')
 elif '산점도' in graph_var.value:
 g = sns.FacetGrid(df_under300.sample(3000, random_state=42), 
 col=col, col_wrap=3, height=3.5)
 g.map(plt.scatter, 'longitude', 'latitude', alpha=0.3, s=5, color='#2196F3')
 else:
 g = sns.FacetGrid(df_under300, col=col, col_wrap=3, height=3.5)
 g.map(sns.kdeplot, 'price', fill=True, color='#2196F3', alpha=0.5)
 
 g.set_titles('{col_name}')
 g.figure.suptitle(f'FacetGrid: {col}별 {graph_var.value}', y=1.02, fontweight='bold')
 plt.tight_layout()
 plt.show()
 
 print(f"\n ▶ FacetGrid 장점: 동일한 축으로 여러 그룹을 비교 가능!")
 print(f" → AI 연결: 모델 성능을 여러 데이터 그룹별로 비교할 때도")
 print(f" FacetGrid를 사용한다 (예: 연령대별 예측 정확도).")

col_var.observe(update_facet, names='value')
graph_var.observe(update_facet, names='value')
display(widgets.VBox([widgets.HBox([col_var, graph_var]), out_facet]))
update_facet()


---

# PART 4: 심화 — 시각화의 미래

> EDA는 학습 **"전"**의 시각화이다. 학습 **"후"**에는 **모델이 왜 그런 예측을 했는지** 시각화해야 한다.

### ▶ 비유: "**의사의 진단 과정**"

> - **EDA** = 진찰·검사 (환자를 파악한다) → 학습 **전**
> - **학습 곡선** = 치료 경과 (치료가 잘 되고 있는가?) → 학습 **중**
> - **XAI** = 진단서 (왜 이 병이라고 판단했는가?) → 학습 **후**
>
> 세 단계 모두 **시각화**로 확인한다. EDA를 마스터하면 나머지도 자연스럽게 이해된다.

| 시각화 도구 | 시점 | 목적 |
|---|---|---|
| 히스토그램, 박스플롯, 히트맵 | 학습 **전** (EDA) | 데이터 품질 점검 |
| **학습 곡선**(Learning Curve) | 학습 **중** | 과적합/과소적합 진단 |
| **SHAP, Grad-CAM, 특성 중요도** | 학습 **후** (XAI) | 모델 해석 |


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V28] XAI 맛보기 ① — 특성 중요도 (Feature Importance)
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V28] 특성 중요도 — 모델이 무엇을 보고 판단하는가?")
print(" ▶ 비유: 입시에서 수능(50%), 내신(30%), 면접(20%)처럼,")
print(" 모델도 각 특성에 '가중치'를 부여한다.")
print(" 특성 중요도는 이 가중치를 시각화한 것이다.")
print("=" * 60)

from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

# Iris 데이터로 간단한 예제
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_iris, y_iris)

# 특성 중요도 시각화
importance = pd.Series(rf.feature_importances_, index=iris.feature_names).sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

importance.plot.barh(color=CVD_SAFE[0], edgecolor='black', ax=ax1)
ax1.set_title('Random Forest 특성 중요도\n→ petal length가 가장 중요하다', fontweight='bold')
ax1.set_xlabel('중요도 (Gini Importance)')

# EDA 상관관계와 비교
for i, feature in enumerate(iris.feature_names):
 r = np.corrcoef(X_iris[feature], y_iris)[0, 1]
 ax2.barh(feature, abs(r), color=CVD_SAFE[2], edgecolor='black')
ax2.set_title('EDA 상관계수 |r| (타겟과의 상관)\n→ EDA에서도 petal이 중요함을 발견!', fontweight='bold')
ax2.set_xlabel('|상관계수|')

plt.tight_layout()
plt.show()

print()
print(" ▶ 비교:")
print(" • EDA 상관 분석: petal length의 |r|이 가장 높다 → 타겟과 강한 관계")
print(" • XAI 특성 중요도: petal length의 Gini 중요도가 가장 높다")
print(" → EDA에서 발견한 것을 모델도 확인해 주었다!")
print()
print(" → AI: SHAP(SHapley Additive exPlanations)은 특성 중요도의 진화 버전이다.")
print(" 각 예측에 대해 '어떤 특성이 얼마나 기여했는지' 개별적으로 설명한다.")
print(" → EU AI Act는 고위험 AI에 이런 설명을 법적으로 요구한다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V29] 학습 곡선 (Learning Curve) — 과적합 진단
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V29] 학습 곡선 — 과적합 vs 과소적합")
print(" ▶ 비유: 시험 공부를 할 때—")
print(" • 과소적합: 교과서만 읽고 문제를 안 풀었다 → 연습 부족")
print(" • 과적합: 기출 답을 외웠다 → 새 문제에 약하다")
print(" • 적정: 원리를 이해하고 다양한 문제를 풀었다 → 새 문제도 잘 푼다")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 시뮬레이션 데이터
epochs = np.arange(1, 101)

# 과소적합
train_loss1 = 1.5 * np.exp(-0.01 * epochs) + 0.8
val_loss1 = 1.5 * np.exp(-0.01 * epochs) + 0.85
axes[0].plot(epochs, train_loss1, color=CVD_SAFE[0], lw=2, label='Train Loss')
axes[0].plot(epochs, val_loss1, color=CVD_SAFE[5], lw=2, label='Val Loss')
axes[0].set_title('과소적합 (Underfitting)\n두 손실 모두 높다', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_ylim(0, 2.5)
axes[0].annotate('둘 다 높다!', xy=(50, 1.0), fontsize=11, color='red',
 fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightyellow'))

# 과적합
train_loss2 = 2.0 * np.exp(-0.05 * epochs) + 0.1
val_loss2 = 2.0 * np.exp(-0.03 * epochs) + 0.3 + 0.005 * epochs
axes[1].plot(epochs, train_loss2, color=CVD_SAFE[0], lw=2, label='Train Loss')
axes[1].plot(epochs, val_loss2, color=CVD_SAFE[5], lw=2, label='Val Loss')
axes[1].set_title('과적합 (Overfitting)\nTrain↓ Val↑ 갈라진다', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].set_ylim(0, 2.5)
axes[1].annotate('갈라진다!', xy=(60, 0.8), fontsize=11, color='red',
 fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightyellow'))

# 적정
train_loss3 = 2.0 * np.exp(-0.04 * epochs) + 0.15
val_loss3 = 2.0 * np.exp(-0.035 * epochs) + 0.2
axes[2].plot(epochs, train_loss3, color=CVD_SAFE[0], lw=2, label='Train Loss')
axes[2].plot(epochs, val_loss3, color=CVD_SAFE[5], lw=2, label='Val Loss')
axes[2].set_title('적정 (Good Fit)\n두 손실이 함께 낮아진다', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].set_ylim(0, 2.5)
axes[2].annotate('함께 수렴!', xy=(50, 0.3), fontsize=11, color='green',
 fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightgreen'))

plt.tight_layout()
plt.show()

print()
print(" ▶ 학습 곡선 읽는 법:")
print(" • Train과 Val 모두 높다 → 과소적합 → 모델 복잡도 ↑ 또는 특성 추가")
print(" • Train↓ Val↑ 갈라짐 → 과적합 → 데이터 ↑, 정규화, 드롭아웃")
print(" • 둘 다 낮고 수렴 → 적정 → 이 상태를 유지!")
print()
print(" → AI: TensorBoard에서 실시간으로 학습 곡선을 모니터링한다.")
print(" 조기 종료(Early Stopping): Val Loss가 올라가기 시작하면 학습을 멈춘다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V30] SHAP 개념 시각화 — 워터폴 차트
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V30] SHAP 워터폴 차트 (개념 시각화)")
print(" ▶ 비유: '이 환자의 심장병 확률이 75%인 이유'를 분석한다.")
print(" 각 특성이 확률을 올렸는지(+), 내렸는지(-)를 보여 준다.")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 6))

features = ['기본 확률', '나이=65', '혈압=High', '운동=Yes', '흡연=No', '콜레스테롤=High', '최종 예측']
values = [50, +12, +8, -5, -3, +13, 75]
colors_bar = ['gray', CVD_SAFE[5], CVD_SAFE[5], CVD_SAFE[2], CVD_SAFE[2], CVD_SAFE[5], CVD_SAFE[0]]

cumulative = [50]
for v in values[1:-1]:
 cumulative.append(cumulative[-1] + v)
cumulative.append(values[-1])

# 워터폴 차트
for i in range(len(features)):
 if i == 0:
 ax.barh(features[i], values[i], color=colors_bar[i], edgecolor='black')
 elif i == len(features) - 1:
 ax.barh(features[i], values[i], color=colors_bar[i], edgecolor='black')
 else:
 left = cumulative[i-1] if values[i] > 0 else cumulative[i]
 width = abs(values[i])
 ax.barh(features[i], width, left=left if values[i] < 0 else cumulative[i-1],
 color=colors_bar[i], edgecolor='black')
 sign = '+' if values[i] > 0 else ''
 ax.text(cumulative[i] + 1, i, f'{sign}{values[i]}%', va='center', fontweight='bold')

ax.set_xlabel('심장병 확률 (%)')
ax.set_title('SHAP 워터폴 차트 (개념 시각화)\n→ 각 특성이 예측에 미친 영향', fontweight='bold', fontsize=13)
ax.axvline(50, color='gray', linestyle=':', alpha=0.5, label='기본 확률')
ax.axvline(75, color='red', linestyle='--', alpha=0.7, label='최종 예측')
ax.legend()
ax.set_xlim(0, 100)

plt.tight_layout()
plt.show()

print()
print(" ▶ SHAP 읽는 법:")
print(" • 빨간 막대(+): 이 특성이 심장병 확률을 높였다")
print(" • 초록 막대(-): 이 특성이 심장병 확률을 낮췄다")
print(" • 기본 확률 50% + 나이(+12) + 혈압(+8) + 운동(-5) + ... = 최종 75%")
print()
print(" → AI: SHAP은 게임 이론의 Shapley Value에 기반한다.")
print(" EU AI Act는 의료·금융 AI에 이런 설명을 법적으로 요구한다.")
print(" → '블랙박스' AI는 법적으로 사용할 수 없게 된다!")


---

## 4.2 고차원 시각화 — **PCA**와 **t-SNE** 맛보기

### ▶ 비유: "**그림자 투영**"

> 3D 물체에 손전등을 비추면 2D 그림자가 생긴다.
> PCA는 **가장 정보가 많은 방향으로** 손전등을 비추는 것이다.
>
> - **PCA**: 분산이 최대인 축을 찾아 투영한다 → 전체 구조 파악
> - **t-SNE**: 가까운 점들을 가까이, 먼 점들을 멀리 배치한다 → 군집 탐색
>
> 둘 다 "100차원을 2차원으로 압축하여 눈으로 볼 수 있게" 한다.

### → AI 연결

- **GPT의 토큰 임베딩**: 50,000차원을 t-SNE로 2D에 시각화하면, 유사한 단어가 모여 있다
- **CNN의 특성 맵**: 이미지 분류 모델의 마지막 층을 PCA로 시각화하면, 같은 클래스끼리 모인다
- **추천 시스템**: 사용자/아이템 임베딩을 t-SNE로 시각화하여 군집을 확인한다


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V31] PCA — 64차원 → 2차원 투영
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V31] PCA & t-SNE 비교")
print(" ▶ 비유: 3D 물체에 손전등을 비추면 2D 그림자가 생긴다.")
print(" PCA는 '가장 정보가 많은 방향'으로 비추는 것이다.")
print("=" * 60)

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits

digits = load_digits()
X_dig, y_dig = digits.data, digits.target

print(f" 원본 데이터: {X_dig.shape[0]}개 × {X_dig.shape[1]}차원")
print(f" → 64차원을 2차원으로 압축한다!")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_dig)

sc1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_dig, cmap='tab10', s=10, alpha=0.6)
axes[0].set_title(f'PCA (분산 설명률: {pca.explained_variance_ratio_.sum():.1%})\n→ 전체 구조를 보여 준다',
 fontweight='bold')
plt.colorbar(sc1, ax=axes[0], label='숫자')

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
X_tsne = tsne.fit_transform(X_dig)

sc2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_dig, cmap='tab10', s=10, alpha=0.6)
axes[1].set_title('t-SNE (perplexity=30)\n→ 군집을 선명하게 분리한다', fontweight='bold')
plt.colorbar(sc2, ax=axes[1], label='숫자')

# 비교 표
comparison = [
 ['PCA', 't-SNE'],
 ['선형 변환', '비선형 변환'],
 ['전역 구조 보존', '지역 구조 보존'],
 ['빠르다', '느리다'],
 ['분산 설명률 있음', '없음'],
]
axes[2].axis('off')
table = axes[2].table(cellText=comparison[1:],
 colLabels=comparison[0],
 rowLabels=['원리', '구조', '속도', '설명률'],
 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)
axes[2].set_title('PCA vs t-SNE 비교', fontweight='bold')

plt.tight_layout()
plt.show()

print()
print(f" ▶ PCA 분산 설명률: {pca.explained_variance_ratio_[0]:.1%} + {pca.explained_variance_ratio_[1]:.1%}")
print(f" → 2개 축으로 전체 분산의 {pca.explained_variance_ratio_.sum():.1%}를 설명한다")
print()
print(" → AI: GPT의 토큰 임베딩(50,000차원)을 t-SNE로 시각화하면,")
print(" 'king', 'queen', 'prince'가 한 군집에 모인다.")
print(" → Word2Vec: king - man + woman ≈ queen")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V32] 인터랙티브 — t-SNE perplexity 조절
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V32] 인터랙티브 — perplexity가 t-SNE에 미치는 영향")
print(" ▶ 비유: perplexity는 '이웃의 범위'를 결정한다.")
print(" 작으면 좁은 동네만, 크면 넓은 지역을 고려한다.")
print("=" * 60)

def plot_tsne_perplexity(perplexity=30, n_samples=500):
 np.random.seed(42)
 idx = np.random.choice(len(X_dig), min(n_samples, len(X_dig)), replace=False)
 X_sub = X_dig[idx]
 y_sub = y_dig[idx]

 tsne_local = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=500)
 X_2d = tsne_local.fit_transform(X_sub)

 fig, ax = plt.subplots(figsize=(8, 6))
 scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_sub, cmap='tab10', s=20, alpha=0.7)
 plt.colorbar(scatter, ax=ax, label='숫자')
 ax.set_title(f't-SNE (perplexity={perplexity}, n={n_samples})', fontweight='bold', fontsize=13)
 plt.tight_layout()
 plt.show()

 print(f" perplexity={perplexity}: ", end="")
 if perplexity < 10:
 print("매우 작은 이웃 → 군집이 잘게 쪼개진다 (과적합 위험)")
 elif perplexity < 50:
 print("적절한 범위 → 군집이 잘 보인다")
 else:
 print("큰 이웃 → 군집이 뭉개진다 (과소적합 위험)")

try:
 widgets.interact(
 plot_tsne_perplexity,
 perplexity=widgets.IntSlider(min=5, max=100, step=5, value=30, description='Perplexity:'),
 n_samples=widgets.IntSlider(min=200, max=1000, step=100, value=500, description='표본 수:')
 )
except:
 plot_tsne_perplexity(30)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V33] → AI 시각화 — 혼동행렬(Confusion Matrix) = 히트맵의 확장
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V33] 혼동행렬 — ML 분류 모델의 성적표")
print(" ▶ 비유: 히트맵은 '상관관계'를 보여 주었다면,")
print(" 혼동행렬은 '모델의 맞힘/틀림'을 보여 준다.")
print(" → EDA 히트맵을 이해하면 혼동행렬도 읽을 수 있다!")
print("=" * 60)

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

# Iris로 간단한 분류
X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 혼동행렬 히트맵
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
 xticklabels=iris.target_names, yticklabels=iris.target_names,
 ax=ax1, linewidths=1, linecolor='white')
ax1.set_title('혼동행렬 (Confusion Matrix)\n→ 대각선이 밝을수록 정확하다', fontweight='bold')
ax1.set_xlabel('예측 (Predicted)')
ax1.set_ylabel('실제 (Actual)')

# 정규화된 혼동행렬
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='viridis',
 xticklabels=iris.target_names, yticklabels=iris.target_names,
 ax=ax2, linewidths=1, linecolor='white', vmin=0, vmax=1)
ax2.set_title('정규화 혼동행렬\n→ 각 클래스별 정확률', fontweight='bold')
ax2.set_xlabel('예측 (Predicted)')
ax2.set_ylabel('실제 (Actual)')

plt.tight_layout()
plt.show()

accuracy = np.diag(cm).sum() / cm.sum()
print(f"\n 전체 정확도: {accuracy:.1%}")
for i, name in enumerate(iris.target_names):
 precision = cm[i,i] / cm[:,i].sum() if cm[:,i].sum() > 0 else 0
 recall = cm[i,i] / cm[i,:].sum() if cm[i,:].sum() > 0 else 0
 print(f" • {name}: 정밀도={precision:.1%}, 재현율={recall:.1%}")
print()
print(" → EDA 히트맵 → 혼동행렬 연결:")
print(" • 둘 다 sns.heatmap()으로 그린다")
print(" • 둘 다 색상으로 '크기'를 표현한다")
print(" • EDA: 변수 간 관계 → 혼동행렬: 예측 정확도")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-A5] 프로젝트 종합 대시보드 — 8개 패널
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "자동차 계기판"
# 속도계, 연료계, 엔진 온도, RPM... 여러 지표를 한 화면에!
# 데이터 분석 대시보드도 마찬가지이다.

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("=" * 70)
print(" 종합 대시보드 — 8개 패널로 한눈에 보는 Airbnb NYC")
print("=" * 70)
print()
print(" ▶ 비유: 자동차 계기판처럼 핵심 지표를 한 화면에 모은다!")
print()

fig, axes = plt.subplots(2, 4, figsize=(20, 10), facecolor='white')

# 1. 가격 분포
sns.histplot(df_under300['price'], bins=40, kde=True, color='#2196F3', ax=axes[0,0])
axes[0,0].axvline(df_under300['price'].median(), color='red', linestyle='--', linewidth=2)
axes[0,0].set_title('① 가격 분포', fontweight='bold')

# 2. 지역별 매물 수
df['neighbourhood_group'].value_counts().plot(kind='bar', color='#FF5722', ax=axes[0,1])
axes[0,1].set_title('② 지역별 매물 수', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. room_type 비율
df['room_type'].value_counts().plot(kind='pie', autopct='%1.1f%%', 
 colors=['#2196F3','#FF5722','#4CAF50','#FF9800'], ax=axes[0,2])
axes[0,2].set_title('③ 숙소 유형 비율', fontweight='bold')
axes[0,2].set_ylabel('')

# 4. 지역별 중앙 가격
df_under300.groupby('neighbourhood_group')['price'].median().sort_values().plot(
 kind='barh', color='#9C27B0', ax=axes[0,3])
axes[0,3].set_title('④ 지역별 중앙 가격', fontweight='bold')

# 5. 리뷰 분포
sns.histplot(df_under300['number_of_reviews'], bins=50, color='#4CAF50', ax=axes[1,0])
axes[1,0].set_title('⑤ 리뷰 수 분포', fontweight='bold')

# 6. 가용일수 분포
sns.histplot(df_under300['availability_365'], bins=50, color='#FF9800', ax=axes[1,1])
axes[1,1].set_title('⑥ 연간 가용일수', fontweight='bold')

# 7. 상관 히트맵 (간소)
num_cols = ['price','minimum_nights','number_of_reviews','availability_365']
corr_small = df_filtered[num_cols].corr()
sns.heatmap(corr_small, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
 ax=axes[1,2], cbar_kws={'shrink': 0.8})
axes[1,2].set_title('⑦ 상관 히트맵', fontweight='bold')

# 8. 가격대별 매물 수
bins_price = [0, 50, 100, 150, 200, 250, 300]
labels_price = ['~50', '51-100', '101-150', '151-200', '201-250', '251-300']
df_under300['price_range'] = pd.cut(df_under300['price'], bins=bins_price, labels=labels_price)
df_under300['price_range'].value_counts().sort_index().plot(kind='bar', color='#00BCD4', ax=axes[1,3])
axes[1,3].set_title('⑧ 가격대별 매물 수', fontweight='bold')
axes[1,3].tick_params(axis='x', rotation=45)

plt.suptitle(' NYC Airbnb 종합 대시보드', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print(" ▶ 대시보드 핵심 인사이트:")
print(" 1. 가격: 오른쪽 편향 (대부분 $100 이하)")
print(" 2. 매물: Manhattan > Brooklyn >> Queens > Bronx > Staten Island")
print(" 3. 유형: 집 전체(52%) > 개인실(45%) > 공유실(3%)")
print(" 4. 리뷰: 대부분 10개 미만 (오른쪽 극단 편향)")
print()
print(" → AI 연결:")
print(" 실무에서는 Streamlit이나 Dash로 이러한 대시보드를")
print(" 웹 앱으로 배포한다. AI 모델의 실시간 예측 결과를")
print(" 대시보드로 모니터링하는 것이 MLOps의 핵심이다.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V34] → AI 시각화 — 결정 경계(Decision Boundary) = 산점도의 확장
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" [V34] 결정 경계 — ML 분류 모델이 '어떻게 구분하는가?'")
print(" ▶ 비유: 운동장에 금을 그어 '이쪽은 A팀, 저쪽은 B팀'으로")
print(" 나누는 것이 결정 경계이다. 산점도에 '경계선'을 그린 것이다.")
print("=" * 60)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# 2D 데이터만 사용 (petal length, petal width)
X_2d = X_iris[['petal length (cm)', 'petal width (cm)']].values
y_2d = y_iris

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models = [
 ('k-NN (k=5)', KNeighborsClassifier(n_neighbors=5)),
 ('SVM (RBF)', SVC(kernel='rbf', gamma='scale')),
 ('Random Forest', RandomForestClassifier(n_estimators=50, random_state=42))
]

for ax, (name, model) in zip(axes, models):
 model.fit(X_2d, y_2d)

 # 결정 경계 그리기
 x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
 y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
 xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
 np.linspace(y_min, y_max, 200))
 Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

 ax.contourf(xx, yy, Z, alpha=0.3, cmap='Set3')
 ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, cmap='tab10', s=20, edgecolors='black', linewidth=0.3)
 ax.set_title(name, fontweight='bold', fontsize=13)
 ax.set_xlabel('petal length')
 ax.set_ylabel('petal width')

plt.tight_layout()
plt.show()

print()
print(" ▶ 결정 경계 비교:")
print(" • k-NN: 경계가 울퉁불퉁하다 → 데이터에 민감 (과적합 위험)")
print(" • SVM: 부드러운 곡선 경계 → 커널 함수의 효과")
print(" • Random Forest: 직각 경계 → 트리 기반 모델의 특성")
print()
print(" → 이것은 EDA 산점도의 확장이다:")
print(" • EDA: 산점도로 '데이터의 분포'를 본다")
print(" • ML: 산점도 위에 '모델의 결정 경계'를 추가한다")
print(" → EDA에서 본 패턴을 모델이 얼마나 잘 학습했는지 확인!")


---

# PART 5: 연습문제 (필수 10 + 보너스 2) + 전체 해답

### ▶ 비유: "운전면허 실기 시험"

> PART 1~2에서 교통법규(이론)와 교습(가이드 실습)을 마쳤다.
> 이제 **혼자서 운전**해 볼 시간이다. 처음에는 어렵지만, 반복하면 자연스러워진다.


---

## 문제

### 문제 1. 앤스콤의 사중주 교훈 서술
앤스콤의 사중주가 주는 교훈을 2~3문장으로 서술하고, 이것이 ML/AI에서 왜 중요한지 한 문장을 추가하라.

### 문제 2. 히스토그램 — `hue`로 그룹 분리
에어비앤비 `df_under300`의 `price` 히스토그램을 `neighbourhood_group`별로 분리하여 그려라. `palette="colorblind"` 사용.

### 문제 3. 박스플롯 + 해석
`room_type`별 가격 박스플롯을 그리고, 각 유형의 중앙값과 IQR을 print하라.

### 문제 4. 바이올린 split
Manhattan과 Brooklyn만 추출하여, `split=True`로 `room_type`별 가격 바이올린 플롯을 그려라.

### 문제 5. 산점도 + 회귀선
`number_of_reviews` vs `price` 산점도에 회귀선을 추가하라. 상관계수도 출력하라.

### 문제 6. 히트맵 삼각형
수치형 변수만 골라 상관 히트맵을 그리되, **상삼각만 마스킹**하라. `cmap="viridis"` 사용.

### 문제 7. 카운트플롯
`room_type`별 매물 수를 `neighbourhood_group` 기준으로 분리하여 그려라.

### 문제 8. 서브플롯 2×2
4가지 그래프(히스토그램, 박스플롯, 산점도, 히트맵)를 2×2 서브플롯에 담아라.

### 문제 9. Plotly 3종
Plotly로 (1) 대화형 산점도, (2) 대화형 박스플롯, (3) 대화형 히스토그램을 그려라.

### 문제 10. EDA 미니 보고서
`df_under300`에 대해 5가지 EDA 체크리스트(형태·타입, 결측값, 분포, 이상치, 상관관계)를 수행하고, 수치를 포함한 관찰을 3개 이상 작성하라.

### 문제 11. (보너스) 페어플롯
`pairplot`으로 수치형 변수 4개의 모든 조합을 시각화하라. `hue`로 그룹을 구분하라.

### 문제 12. (보너스) FacetGrid
`FacetGrid`로 `neighbourhood_group`별 가격 히스토그램을 격자로 나열하라.


---

## 전체 해답

> 각 해답을 클릭하면 코드가 펼쳐진다.

<details>
<summary>▶ 해답 1 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V35] 해답 1. 앤스콤의 사중주 교훈
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 1: 앤스콤의 사중주 교훈")
print("=" * 60)
print()
print(" 앤스콤의 사중주는 평균, 표준편차, 상관계수 등 요약 통계량이")
print(" 동일하더라도 데이터의 분포는 완전히 다를 수 있음을 보여 준다.")
print(" 따라서 통계량만으로 데이터를 판단하면 안 되며, 반드시 시각화를")
print(" 통해 분포의 형태를 확인해야 한다.")
print()
print(" ML/AI에서의 중요성:")
print(" 데이터의 분포를 확인하지 않고 모델을 학습시키면,")
print(" 비선형 패턴이나 이상치를 놓쳐 성능이 크게 저하될 수 있다.")
print(" → '데이터 중심 AI'의 핵심 원칙이다.")

```

</details>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-A6] → AI 시각화 — 활성화 함수(Activation Function)
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "문지기(Gatekeeper)"
# 뉴런에 들어온 신호를 "통과시킬지 말지" 결정하는 함수이다.
# 마치 학교 정문 앞 경비 아저씨처럼:
# - Step 함수: 통과 or 불통과 (0 or 1)
# - Sigmoid: 조금 통과, 많이 통과 (0~1 사이)
# - ReLU: 0 이하는 차단, 0 이상은 그대로 통과
# - GELU: GPT가 사용하는 '부드러운 ReLU'

import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print(" 활성화 함수 — 뉴런의 '문지기'")
print("=" * 70)
print()
print(" ▶ 비유: 학교 정문 경비 아저씨")
print(" ─────────────────────────────────────────────")
print(" Step: '학생증 있으면 통과, 없으면 차단' (이진 판단)")
print(" Sigmoid: '학생증 상태에 따라 0~100% 통과' (확률적)")
print(" ReLU: '음수 신호는 무조건 차단, 양수는 그대로' (간단)")
print(" GELU: 'ReLU보다 부드럽게 판단' (GPT가 사용!)")
print()

x = np.linspace(-4, 4, 500)

# 활성화 함수들
step = np.where(x >= 0, 1, 0)
sigmoid = 1 / (1 + np.exp(-x))
relu = np.maximum(0, x)
leaky_relu = np.where(x > 0, x, 0.1 * x)
tanh = np.tanh(x)
gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor='white')

funcs = [(step, 'Step (계단)', '#607D8B', '초기 퍼셉트론 (1958)'),
 (sigmoid, 'Sigmoid (시그모이드)', '#FF5722', '전통 신경망'),
 (tanh, 'Tanh (하이퍼볼릭 탄젠트)', '#9C27B0', 'LSTM (1997)'),
 (relu, 'ReLU', '#2196F3', 'AlexNet (2012) 이후 표준'),
 (leaky_relu, 'Leaky ReLU', '#4CAF50', 'ReLU 개선'),
 (gelu, 'GELU', '#F44336', 'GPT, BERT (2018~)')]

for idx, (y, name, color, era) in enumerate(funcs):
 ax = axes[idx // 3, idx % 3]
 ax.plot(x, y, color=color, linewidth=2.5)
 ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
 ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
 ax.set_title(f'{name}\n({era})', fontweight='bold', fontsize=11)
 ax.set_xlim(-4, 4)
 ax.grid(True, alpha=0.2)
 # 도함수 (그림자)
 dy = np.gradient(y, x)
 ax.fill_between(x, 0, dy, alpha=0.1, color=color)
 ax.plot(x, dy, color=color, linewidth=1, linestyle=':', alpha=0.5, label='도함수')
 ax.legend(fontsize=8, loc='upper left')

plt.suptitle('활성화 함수(Activation Function) — 뉴런의 문지기 진화사', 
 fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print(" ▶ 활성화 함수 진화 연대기:")
print(" ┌───────┬─────────────┬───────────────────────────────┐")
print(" │ 연도 │ 함수 │ 특징 │")
print(" ├───────┼─────────────┼───────────────────────────────┤")
print(" │ 1958 │ Step │ 0 or 1, 미분 불가 │")
print(" │ 1980s │ Sigmoid │ 매끄럽지만 기울기 소실 문제 │")
print(" │ 1997 │ Tanh │ Sigmoid 개선, LSTM에 사용 │")
print(" │ 2012 │ ReLU │ 단순하고 빠름, 현재 표준 │")
print(" │ 2015 │ Leaky ReLU │ 'Dying ReLU' 문제 해결 │")
print(" │ 2018 │ GELU │ GPT·BERT의 선택 │")
print(" └───────┴─────────────┴───────────────────────────────┘")
print()
print(" → AI 연결:")
print(" GPT-4가 GELU를 사용하는 이유:")
print(" → ReLU는 0에서 '꺾인다' (미분 불연속)")
print(" → GELU는 0 근처에서 '부드럽게' 전환된다")
print(" → 이 부드러움이 언어 모델의 세밀한 표현력을 높인다!")


<details>
<summary>▶ 해답 2 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V36] 해답 2. 히스토그램 hue
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 2: hue로 그룹 분리된 히스토그램")
print(" ▶ hue를 사용하면 한 그래프에서 여러 그룹을 비교할 수 있다.")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 6))
if 'neighbourhood_group' in df_under300.columns:
 sns.histplot(df_under300, x='price', hue='neighbourhood_group',
 palette='colorblind', kde=True, ax=ax, alpha=0.5, edgecolor='white')
 ax.set_title('지역별 가격 분포 (hue 분리)', fontweight='bold')
 ax.set_xlabel('가격 ($)')
 ax.set_ylabel('빈도')
plt.tight_layout()
plt.show()

print(" ▶ 관찰: Manhattan이 오른쪽으로 가장 넓게 펼쳐져 있다 → 고가 매물이 많다")

```

</details>

<details>
<summary>▶ 해답 3 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V37] 해답 3. 박스플롯 + 해석
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 3: 숙소 유형별 박스플롯 + 수치 해석")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 6))
if 'room_type' in df_under300.columns:
 sns.boxplot(x='room_type', y='price', data=df_under300,
 palette='colorblind', ax=ax)
 ax.set_title('숙소 유형별 가격 박스플롯', fontweight='bold')
 ax.set_xlabel('숙소 유형')
 ax.set_ylabel('가격 ($)')

 for rt in df_under300['room_type'].unique():
 subset = df_under300[df_under300['room_type'] == rt]['price']
 Q1 = subset.quantile(0.25)
 Q3 = subset.quantile(0.75)
 print(f" • {rt}: 중앙값=${subset.median():.0f}, Q1=${Q1:.0f}, Q3=${Q3:.0f}, IQR=${Q3-Q1:.0f}")

plt.tight_layout()
plt.show()

```

</details>

<details>
<summary>▶ 해답 4 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V38] 해답 4. 바이올린 split
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 4: Manhattan vs Brooklyn 바이올린 플롯")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 6))
if all(c in df_under300.columns for c in ['neighbourhood_group', 'room_type']):
 two_boroughs = df_under300[df_under300['neighbourhood_group'].isin(['Manhattan', 'Brooklyn'])]
 two_rooms = two_boroughs[two_boroughs['room_type'].isin(['Entire home/apt', 'Private room'])]
 sns.violinplot(x='neighbourhood_group', y='price', hue='room_type',
 data=two_rooms, split=True, palette='colorblind', ax=ax, inner='quartile', cut=0)
 ax.set_title('Manhattan vs Brooklyn — 숙소 유형별 가격 (split)', fontweight='bold')
plt.tight_layout()
plt.show()

print(" ▶ split=True: 한 바이올린의 좌우가 다른 그룹을 나타낸다")

```

</details>

<details>
<summary>▶ 해답 5 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V39] 해답 5. 산점도 + 회귀선
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 5: 리뷰 수 vs 가격 산점도 + 회귀선")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 6))
if 'number_of_reviews' in df_under300.columns:
 sample = df_under300.sample(min(1000, len(df_under300)), random_state=42)
 sns.regplot(x='number_of_reviews', y='price', data=sample,
 scatter_kws={'alpha': 0.3, 's': 20, 'color': CVD_SAFE[0]},
 line_kws={'color': 'red', 'lw': 2}, ax=ax)
 r = sample['number_of_reviews'].corr(sample['price'])
 ax.set_title(f'리뷰 수 vs 가격 (r = {r:.3f})', fontweight='bold')
 print(f" 상관계수 r = {r:.3f}")
 print(f" → {'약한' if abs(r)<0.3 else '중간' if abs(r)<0.5 else '강한'} {'양의' if r>0 else '음의'} 상관")
plt.tight_layout()
plt.show()

```

</details>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-W6] 인터랙티브: PCA 주성분 수에 따른 설명 분산
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "사진 압축"
# 원본 사진 = 100% 정보 (파일 크기 큼)
# JPEG 압축 = 95% 정보 (파일 크기 작음, 거의 차이 없음)
# 너무 많이 압축 = 50% 정보 (화질 저하!)
# PCA = 데이터의 JPEG 압축이다!

import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

print("=" * 70)
print(" 인터랙티브: PCA 주성분 수 탐색기")
print("=" * 70)
print()
print(" ▶ 비유: JPEG 압축")
print(" ─────────────────────────────────────────────")
print(" 주성분 64개(원본) = 무압축 사진 (정보 100%)")
print(" 주성분 10개 = 중간 압축 (정보 ~85%)")
print(" 주성분 2개 = 강한 압축 (정보 ~30%)")
print(" → 정보를 최대한 보존하면서 차원을 줄이는 것이 목표!")
print()

digits = load_digits()
X_scaled = StandardScaler().fit_transform(digits.data)
pca_full = PCA().fit(X_scaled)

out_pca = widgets.Output()
n_comp_slider = widgets.IntSlider(value=10, min=2, max=64, step=1,
 description='주성분 수:', style={'description_width': '80px'},
 layout=widgets.Layout(width='450px'))

def update_pca(*args):
 with out_pca:
 clear_output(wait=True)
 n = n_comp_slider.value
 explained = np.cumsum(pca_full.explained_variance_ratio_)
 
 fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='white')
 
 # 누적 설명 분산
 ax1.plot(range(1, 65), explained, 'b-', linewidth=2)
 ax1.axvline(n, color='red', linewidth=2, linestyle='--')
 ax1.axhline(explained[n-1], color='red', linewidth=1, linestyle=':', alpha=0.5)
 ax1.fill_between(range(1, n+1), 0, explained[:n], alpha=0.2, color='blue')
 ax1.scatter([n], [explained[n-1]], s=150, c='red', zorder=5)
 ax1.annotate(f'{n}개 → {explained[n-1]:.1%}', (n, explained[n-1]),
 textcoords='offset points', xytext=(10, 10), fontsize=12,
 fontweight='bold', color='red')
 ax1.set_xlabel('주성분 수')
 ax1.set_ylabel('누적 설명 분산')
 ax1.set_title('누적 설명 분산 비율', fontweight='bold')
 ax1.set_ylim(0, 1.05)
 ax1.axhline(0.95, color='gray', linestyle='--', alpha=0.3, label='95% 기준선')
 ax1.legend(fontsize=9)
 
 # 2D 투영
 pca_2d = PCA(n_components=2).fit_transform(X_scaled)
 scatter = ax2.scatter(pca_2d[:, 0], pca_2d[:, 1], c=digits.target,
 cmap='tab10', s=10, alpha=0.5)
 ax2.set_title(f'2D 투영 (설명 분산: {explained[1]:.1%})', fontweight='bold')
 ax2.set_xlabel('PC1')
 ax2.set_ylabel('PC2')
 plt.colorbar(scatter, ax=ax2, label='숫자 (0~9)')
 
 plt.suptitle(f'PCA: 64차원 → {n}차원 (정보 {explained[n-1]:.1%} 보존)', 
 fontsize=13, fontweight='bold')
 plt.tight_layout()
 plt.show()
 
 pct = explained[n-1]
 if pct >= 0.95:
 quality = "✅ 95% 이상! 거의 정보 손실 없이 차원 축소 성공"
 elif pct >= 0.80:
 quality = "※ 80~95%: 대부분의 ML 작업에 충분"
 else:
 quality = "❌ 80% 미만: 정보 손실 과다! 주성분을 늘려야 한다"
 print(f"\n {quality}")
 print(f"\n → AI 연결: GPT의 임베딩(embedding)도 일종의 차원 축소이다.")
 print(f" 수만 개의 단어를 768~16384차원의 벡터로 '압축'하여 표현한다.")
 print(f" PCA를 이해하면 임베딩의 원리를 직관적으로 파악할 수 있다!")

n_comp_slider.observe(update_pca, names='value')
display(widgets.VBox([n_comp_slider, out_pca]))
update_pca()


<details>
<summary>▶ 해답 6 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V40] 해답 6. 히트맵 삼각형
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 6: 상관 히트맵 (상삼각 마스킹)")
print("=" * 60)

num_cols = df_under300.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) >= 2:
 corr = df_under300[num_cols].corr()
 mask = np.triu(np.ones_like(corr, dtype=bool))

 fig, ax = plt.subplots(figsize=(10, 8))
 sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='viridis',
 center=0, linewidths=1, linecolor='white', ax=ax, vmin=-1, vmax=1)
 ax.set_title('수치형 변수 상관 히트맵 (하삼각)', fontweight='bold')
 plt.tight_layout()
 plt.show()

```

</details>

<details>
<summary>▶ 해답 7 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V41] 해답 7. 카운트플롯
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 7: 숙소 유형별 × 지역별 카운트플롯")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 6))
if all(c in df_under300.columns for c in ['room_type', 'neighbourhood_group']):
 sns.countplot(x='neighbourhood_group', hue='room_type', data=df_under300,
 palette='colorblind', ax=ax, edgecolor='black')
 ax.set_title('지역별 × 숙소유형별 매물 수', fontweight='bold')
 ax.legend(title='숙소 유형')
plt.tight_layout()
plt.show()

```

</details>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# [V-W5] 인터랙티브: 임계값 조절 → ROC 곡선 이해
# ═══════════════════════════════════════════════════════════════
# ▶ 비유: "스팸 필터의 민감도"
# 임계값을 낮추면: 스팸을 거의 다 잡지만, 정상 메일도 스팸함에!
# 임계값을 높이면: 정상 메일은 안전하지만, 스팸이 받은편지함에!

import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, confusion_matrix

print("=" * 70)
print(" 인터랙티브: 분류 임계값(Threshold)과 ROC 곡선")
print("=" * 70)
print()
print(" ▶ 비유: 스팸 필터의 민감도 다이얼")
print(" ─────────────────────────────────────────────")
print(" 임계값 ↓ (민감) → 스팸 잘 잡음, 정상메일도 스팸으로 (오탐↑)")
print(" 임계값 ↑ (둔감) → 정상메일 안전, 스팸 놓침 (미탐↑)")
print(" → '최적 지점'을 찾는 것이 ML 분류의 핵심이다!")
print()

# 데이터 생성 & 모델 학습
np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=2, n_redundant=0,
 n_informative=2, random_state=42, n_clusters_per_class=1)
lr = LogisticRegression(random_state=42).fit(X, y)
y_prob = lr.predict_proba(X)[:, 1]
fpr_all, tpr_all, thresholds_all = roc_curve(y, y_prob)
roc_auc = auc(fpr_all, tpr_all)

out_roc = widgets.Output()
thresh_slider = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05,
 description='임계값:', style={'description_width': '70px'},
 layout=widgets.Layout(width='450px'))

def update_roc(*args):
 with out_roc:
 clear_output(wait=True)
 th = thresh_slider.value
 y_pred = (y_prob >= th).astype(int)
 cm = confusion_matrix(y, y_pred)
 
 fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5), facecolor='white')
 
 # 1. 확률 분포 + 임계값
 ax1.hist(y_prob[y==0], bins=30, alpha=0.5, color='#2196F3', label='Class 0 (정상)', density=True)
 ax1.hist(y_prob[y==1], bins=30, alpha=0.5, color='#FF5722', label='Class 1 (스팸)', density=True)
 ax1.axvline(th, color='black', linewidth=2.5, linestyle='--', label=f'임계값={th:.2f}')
 ax1.fill_betweenx([0, 5], 0, th, alpha=0.05, color='blue')
 ax1.fill_betweenx([0, 5], th, 1, alpha=0.05, color='red')
 ax1.set_title('예측 확률 분포', fontweight='bold')
 ax1.legend(fontsize=8)
 ax1.set_xlabel('예측 확률')
 
 # 2. ROC 곡선
 ax2.plot(fpr_all, tpr_all, 'b-', linewidth=2, label=f'AUC = {roc_auc:.3f}')
 ax2.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='무작위')
 # 현재 임계값 위치
 fpr_now = cm[0,1] / (cm[0,0] + cm[0,1]) if (cm[0,0]+cm[0,1])>0 else 0
 tpr_now = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0
 ax2.scatter([fpr_now], [tpr_now], s=200, c='red', zorder=5, marker='')
 ax2.annotate(f'현재 ({fpr_now:.2f}, {tpr_now:.2f})', 
 (fpr_now, tpr_now), textcoords='offset points',
 xytext=(10, -15), fontsize=10, color='red', fontweight='bold')
 ax2.set_title('ROC 곡선', fontweight='bold')
 ax2.set_xlabel('FPR (오탐률)')
 ax2.set_ylabel('TPR (재현율)')
 ax2.legend(fontsize=9)
 
 # 3. 혼동 행렬
 im = ax3.imshow(cm, cmap='Blues')
 ax3.set_xticks([0, 1])
 ax3.set_yticks([0, 1])
 ax3.set_xticklabels(['정상 예측', '스팸 예측'])
 ax3.set_yticklabels(['실제 정상', '실제 스팸'])
 for i in range(2):
 for j in range(2):
 label_map = {(0,0): 'TN', (0,1): 'FP', (1,0): 'FN', (1,1): 'TP'}
 ax3.text(j, i, f'{label_map[(i,j)]}\n{cm[i,j]}', 
 ha='center', va='center', fontsize=14, fontweight='bold',
 color='white' if cm[i,j] > cm.max()/2 else 'black')
 ax3.set_title('혼동 행렬', fontweight='bold')
 
 plt.suptitle(f'임계값 = {th:.2f}', fontsize=14, fontweight='bold')
 plt.tight_layout()
 plt.show()
 
 acc = (cm[0,0]+cm[1,1]) / cm.sum()
 prec = cm[1,1]/(cm[0,1]+cm[1,1]) if (cm[0,1]+cm[1,1])>0 else 0
 rec = cm[1,1]/(cm[1,0]+cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0
 
 print(f"\n 정확도: {acc:.1%} | 정밀도: {prec:.1%} | 재현율: {rec:.1%}")
 print(f" → AI 연결: 의료 AI는 재현율(놓치지 않기)을,")
 print(f" 스팸 필터는 정밀도(오탐 줄이기)를 더 중시한다!")

thresh_slider.observe(update_roc, names='value')
display(widgets.VBox([thresh_slider, out_roc]))
update_roc()


<details>
<summary>▶ 해답 8 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V42] 해답 8. 서브플롯 2×2
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 8: 4가지 그래프 2×2 서브플롯")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 히스토그램
sns.histplot(df_under300['price'], bins=40, kde=True, color=CVD_SAFE[0], ax=axes[0,0], edgecolor='white')
axes[0,0].set_title('① 히스토그램 + KDE', fontweight='bold')

# 박스플롯
if 'neighbourhood_group' in df_under300.columns:
 sns.boxplot(x='neighbourhood_group', y='price', data=df_under300, palette='colorblind', ax=axes[0,1])
 axes[0,1].tick_params(axis='x', rotation=30)
axes[0,1].set_title('② 박스플롯', fontweight='bold')

# 산점도
if all(c in df_under300.columns for c in ['latitude', 'longitude']):
 s = df_under300.sample(min(1000, len(df_under300)), random_state=42)
 axes[1,0].scatter(s['longitude'], s['latitude'], c=s['price'], cmap='viridis', s=5, alpha=0.5)
axes[1,0].set_title('③ 산점도', fontweight='bold')

# 히트맵
if len(num_cols) >= 2:
 sns.heatmap(df_under300[num_cols[:5]].corr(), annot=True, fmt='.2f', cmap='viridis',
 ax=axes[1,1], linewidths=0.5, center=0)
axes[1,1].set_title('④ 히트맵', fontweight='bold')

plt.tight_layout()
plt.show()

```

</details>

<details>
<summary>▶ 해답 9 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V43] 해답 9. Plotly 3종
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 9: Plotly 대화형 그래프 3종")
print("=" * 60)

try:
 import plotly.express as px
 sample = df_under300.sample(min(500, len(df_under300)), random_state=42)

 # 대화형 산점도
 if 'neighbourhood_group' in sample.columns:
 fig1 = px.scatter(sample, x='longitude', y='latitude', color='neighbourhood_group',
 size='price', title='Plotly 대화형 산점도',
 color_discrete_sequence=px.colors.qualitative.Safe)
 fig1.show()

 # 대화형 박스플롯
 if 'neighbourhood_group' in sample.columns:
 fig2 = px.box(sample, x='neighbourhood_group', y='price',
 title='Plotly 대화형 박스플롯',
 color_discrete_sequence=px.colors.qualitative.Safe)
 fig2.show()

 # 대화형 히스토그램
 fig3 = px.histogram(sample, x='price', nbins=40, title='Plotly 대화형 히스토그램',
 color_discrete_sequence=[px.colors.qualitative.Safe[0]])
 fig3.show()
 print(" ✅ Plotly 3종 생성 완료")
except ImportError:
 print(" ※ Plotly 미설치: pip install plotly")

```

</details>

<details>
<summary>▶ 해답 10 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V44] 해답 10. EDA 미니 보고서
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 10: EDA 미니 보고서")
print("=" * 60)

print(f"\n 1⃣ 형태·타입")
print(f" • 크기: {df_under300.shape[0]:,}행 × {df_under300.shape[1]}열")
print(f" • 타입: {dict(df_under300.dtypes.value_counts())}")

print(f"\n 2⃣ 결측값")
missing = df_under300.isnull().sum()
print(f" • 총 결측: {missing.sum()}개")
if missing.sum() > 0:
 for col in missing[missing > 0].index:
 print(f" - {col}: {missing[col]}개 ({missing[col]/len(df_under300)*100:.1f}%)")

print(f"\n 3⃣ 분포")
print(f" • 가격 왜도: {df_under300['price'].skew():.2f}")
print(f" • 가격 첨도: {df_under300['price'].kurtosis():.2f}")

print(f"\n 4⃣ 이상치 (IQR 기준)")
Q1_p = df_under300['price'].quantile(0.25)
Q3_p = df_under300['price'].quantile(0.75)
IQR_p = Q3_p - Q1_p
outliers = df_under300[(df_under300['price'] < Q1_p - 1.5*IQR_p) |
 (df_under300['price'] > Q3_p + 1.5*IQR_p)]
print(f" • 가격 이상치: {len(outliers)}개 ({len(outliers)/len(df_under300)*100:.1f}%)")

print(f"\n 5⃣ 상관관계")
if 'number_of_reviews' in df_under300.columns:
 r = df_under300['price'].corr(df_under300['number_of_reviews'])
 print(f" • price ↔ number_of_reviews: r = {r:.3f}")

print(f"\n ▶ 관찰:")
print(f" 1. 가격은 오른쪽 꼬리 분포(왜도={df_under300['price'].skew():.2f})로 로그 변환이 권장된다.")
print(f" 2. 전체 매물의 50%가 ${df_under300['price'].median():.0f} 이하이다.")
if 'neighbourhood_group' in df_under300.columns:
 top_region = df_under300.groupby('neighbourhood_group')['price'].median().idxmax()
 top_price = df_under300.groupby('neighbourhood_group')['price'].median().max()
 print(f" 3. {top_region}의 중앙 가격 ${top_price:.0f}이 가장 높다.")

```

</details>

<details>
<summary>▶ 해답 11 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V45] 해답 11. (보너스) 페어플롯
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 11: 페어플롯 (모든 수치형 변수 조합)")
print(" ▶ 비유: 모든 친구 쌍의 사진을 찍어 한 앨범에 모은 것이다.")
print("=" * 60)

pair_cols = [c for c in ['price', 'minimum_nights', 'number_of_reviews'] if c in df_under300.columns]
if len(pair_cols) >= 2 and 'neighbourhood_group' in df_under300.columns:
 sample_pair = df_under300.sample(min(500, len(df_under300)), random_state=42)
 g = sns.pairplot(sample_pair[pair_cols + ['neighbourhood_group']],
 hue='neighbourhood_group', palette='colorblind',
 plot_kws={'alpha': 0.4, 's': 15}, diag_kind='kde')
 g.figure.suptitle('페어플롯 — 모든 수치형 변수 조합', fontweight='bold', y=1.02)
 plt.show()
 print(" ▶ 대각선: 각 변수의 KDE, 비대각선: 변수 쌍의 산점도")

```

</details>

<details>
<summary>▶ 해답 12 (클릭하여 펼치기)</summary>

```python
# ═══════════════════════════════════════════════════════════════
# [V46] 해답 12. (보너스) FacetGrid
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print(" ✅ 해답 12: FacetGrid — 지역별 가격 히스토그램 격자")
print(" ▶ 비유: 같은 사진을 5가지 필터로 찍어 비교하는 것이다.")
print("=" * 60)

if 'neighbourhood_group' in df_under300.columns:
 sample_facet = df_under300.sample(min(2000, len(df_under300)), random_state=42)
 g = sns.FacetGrid(sample_facet, col='neighbourhood_group', col_wrap=3,
 height=4, sharex=True, sharey=True)
 g.map(sns.histplot, 'price', bins=30, color=CVD_SAFE[0], edgecolor='white', kde=True)
 g.figure.suptitle('지역별 가격 분포 FacetGrid', fontweight='bold', y=1.02)
 plt.show()
 print(" ▶ FacetGrid: 같은 그래프를 범주별로 자동 반복 → 비교가 쉽다")

```

</details>

---

# 5주차 핵심 정리

## "왜?"에 대한 답 + AI 연결

| 개념 | 도구 수준 | 전문가 수준 ("왜?") | 최신 AI 연결 |
|---|---|---|---|
| **시각화** | 그래프를 그린다 | 숫자는 거짓말한다 → 시각으로 확인 | 학습 곡선, SHAP, Grad-CAM |
| **분포** | 히스토그램을 그린다 | ML 알고리즘이 정규분포를 가정한다 | **배치 정규화**(BatchNorm), GPT **LayerNorm** |
| **이상치** | 제거한다 | "항상" 제거? → 사기 탐지에서는 답이다 | **오토인코더**, Isolation Forest |
| **상관관계** | 히트맵을 그린다 | 상관 ≠ 인과, 다중공선성 위험 | **어텐션**(Attention) = 상관의 확장 |
| **색맹 배려** | 예쁜 색을 쓴다 | 전문적 소양 + 접근성 | AI 윤리, EU AI Act |
| **EDA** | 데이터를 살펴본다 | ML 파이프라인의 필수 관문 | **데이터 중심 AI**, MLOps |
| **XAI** | (6주차 예고) | 모델의 "왜?"를 설명 | SHAP, Grad-CAM, EU 규제 |
| **차원 축소** | (6주차 예고) | 고차원을 2D로 압축 | t-SNE로 **GPT 임베딩** 시각화 |

---

## 비유 총정리

| 개념 | 비유 |
|---|---|
| EDA | **건강검진** — 데이터를 "진찰"하는 과정 |
| 통계량만 보기 | **전화 진료** — 대면(시각화)이 훨씬 정확하다 |
| 앤스콤의 사중주 | **쌍둥이 건강검진** — 수치는 같지만 X-ray는 다르다 |
| 분포 | **시험 점수** — 난이도에 따라 모양이 달라진다 |
| 이상치 | **190cm 학생** — 오류인지 보물인지 판단해야 한다 |
| 상관관계 | **친구 관계도** — 가까운 것과 영향을 주는 것은 다르다 |
| 색맹 안전 팔레트 | **경사로**(ramp) — 유니버설 디자인 |
| bins 조절 | **현미경 배율** — 세밀도를 조절한다 |
| PCA/t-SNE | **그림자 투영** — 3D를 2D로 압축한다 |
| 학습 곡선 | **시험 공부** — 기출 암기(과적합) vs 원리 이해(적정) |
| SHAP | **진단서** — "왜 이 병이라고 판단했는가?" |
| 결정 경계 | **운동장 금 긋기** — 이쪽은 A팀, 저쪽은 B팀 |

---

## 다음 주 예고: 6주차 — ML 워크플로우

> 이번 주 EDA에서 발견한 패턴을 **모델이 자동으로 학습**한다.
> - 분류(Classification): "이 매물의 지역은?" → **로지스틱 회귀, 결정 트리**
> - 회귀(Regression): "적정 가격은 얼마?" → **선형 회귀, 랜덤 포레스트**
> - 평가: **혼동행렬, ROC 곡선** → 이번 주 히트맵의 확장판이다!
